# Results

Downloads the per-station results logged to W&B (TOK runs from `final_experiment_CNN`, baseline runs from `Final Linear Regression`) and makes the comparison figures against ARSO's forecasts. Figures are saved to `figures/`.

In [ ]:
import os
import pandas as pd

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
import wandb
import numpy as np
import matplotlib.pyplot as plt
# Initialize wandb API
api = wandb.Api()

# Get all runs from the model_experiment project
my_project = "samotorcan-leon-/final_experiment_CNN"
runs = api.runs(my_project)  # Updated with correct username format

# Initialize lists to store run information
run_data = []

for run in runs:
    # Extract run information
    run_info = {
        'id': run.id,
        'name': run.name,
        'station_id': run.config.get('station_id', None),
        'state': run.state,
        'config': dict(run.config),
        'summary': dict(run.summary),
        'created_at': run.created_at,
        'duration': run.summary.get('_runtime', 0),
        'tags': run.tags,
    }
    print(run.name)
    run_info['artifacts'] = run.logged_artifacts()

    # Extract model parameters related to weights and biases
    config = run.config
    run_info['hidden_dim'] = config.get('TOK.hidden_dim', config.get('hidden_dim', None))
    run_info['prediction_layer_hidden_dim'] = config.get('TOK.prediction_layer_hidden_dim', config.get('prediction_layer_hidden_dim', None))
    run_info['dropout_rate'] = config.get('TOK.dropout_rate', config.get('dropout_rate', None))
    run_info['learning_rate'] = config.get('lr', config.get('learning_rate', None))
    run_info['weight_decay'] = config.get('weight_decay', None)
    run_info['use_temporal_blocks'] = config.get('TOK.use_temporal_blocks', config.get('use_temporal_blocks', None))
    run_info['block_hours'] = config.get('TOK.block_hours', config.get('block_hours', None))
    
    run_data.append(run_info)

# Convert to DataFrame for easier analysis
runs_df = pd.DataFrame(run_data)

print(f"Loaded {len(runs_df)} runs from model_experiment project")
print(f"Run states: {runs_df['state'].value_counts().to_dict()}")
print(f"Columns available: {list(runs_df.columns)}")


In [ ]:

def download_metrics_table(run_artifacts, api, project_name, file_name_contains):
    import os
    """
    Download and read metrics table artifact into a pandas DataFrame
    
    Args:
        run_artifacts: List of artifacts from a wandb run
        api: wandb API instance
        project_name: Name of the wandb project

        
    Returns:
        pd.DataFrame: The metrics table data
    """
    for artifact in run_artifacts:
        if file_name_contains in artifact.name:
            #print(f"Found metrics table: {artifact.name}")
            # Download the artifact
            artifact_obj = api.artifact(f"{project_name}/{artifact.name}")
            artifact_dir = artifact_obj.download()
            #print(artifact_dir)
            # Find the table file in the downloaded directory
            for file in os.listdir(artifact_dir):
                if file.endswith('.table.json'):
                    table_path = os.path.join(artifact_dir, file)
                    # Read the wandb table file
                    with open(table_path, 'r') as f:
                        import json
                        table_data = json.load(f)
                    
                    # Convert to DataFrame
                    columns = table_data['columns']
                    data = table_data['data']
                    df = pd.DataFrame(data, columns=columns)
                    return df
    
    return None

# Function to download metrics table for a single run
def download_run_metrics(run_row, file_name_contains):
    """Download metrics table for a single run and return the DataFrame"""
    try:
        metrics_df = download_metrics_table(run_row['artifacts'], api, my_project, file_name_contains)
        return metrics_df
    except Exception as e:
        print(f"Error downloading metrics for run {run_row['name']}: {e}")
        return None

# Download metrics tables for all runs
print("Downloading metrics tables for all runs...")
from tqdm import tqdm
tqdm.pandas()

runs_df['metrics_table'] = runs_df.progress_apply(lambda row: download_run_metrics(row, 'metrics_table'), axis=1)
runs_df['mse_scores'] = runs_df.progress_apply(lambda row: download_run_metrics(row, 'mse_scores'), axis=1)
runs_df['arso_predictions_df'] = runs_df.progress_apply(lambda row: download_run_metrics(row, 'arso_predictions_df'), axis=1)
runs_df['model_predictions_df'] = runs_df.progress_apply(lambda row: download_run_metrics(row, 'model_predictions_df'), axis=1)
runs_df['actuals_df'] = runs_df.progress_apply(lambda row: download_run_metrics(row, 'actuals_df'), axis=1)

# Check results
successful_downloads = runs_df['metrics_table'].notna().sum()
print(f"Successfully downloaded {successful_downloads} out of {len(runs_df)} metrics tables")
        

In [ ]:
runs_df_backup = runs_df.copy()
#runs_df_backup.to_parquet("runs_df_backup.parquet")

In [ ]:
pd.to_datetime(runs_df.iloc[0]['arso_predictions_df'].reset_index()['index'], unit='ms').to_numpy()

In [ ]:
# Loop through each row in runs_df
for i, row in runs_df.iterrows():
    metrics_table = row['metrics_table']
    model_pred = row['model_predictions_df']
    arso_pred = row['arso_predictions_df'] 
    actuals = row['actuals_df']

    # Calculate RMSE for each hour
    model_rmse = []
    arso_rmse = []

    for hour in range(18):  # Assuming 18 hours of predictions
        hour_str = str(hour)
        if hour_str in model_pred.columns and hour_str in arso_pred.columns and hour_str in actuals.columns:
            model_rmse.append(np.sqrt(np.mean((model_pred[hour_str].to_numpy() - actuals[hour_str].to_numpy())**2)))
            arso_rmse.append(np.sqrt(np.mean((arso_pred[hour_str].to_numpy() - actuals[hour_str].to_numpy())**2)))
    #print(model_rmse)
    # Add RMSE columns to metrics table
    metrics_table['Model RMSE'] = model_rmse
    metrics_table['ARSO RMSE'] = arso_rmse

    # Update the metrics table in runs_df
    runs_df.at[i, 'metrics_table'] = metrics_table

# Display first metrics table as example
print(runs_df.query("station_id==9300").iloc[0]['station_id'])
runs_df.query("station_id==9300")['metrics_table'].iloc[0]

In [ ]:
runs_df = runs_df.dropna(subset=['metrics_table'])

In [ ]:
runs_df['model_predictions_df'].iloc[0].shape, runs_df['arso_predictions_df'].iloc[0].shape, runs_df['actuals_df'].iloc[0].shape

In [ ]:
df_combined.columns

In [ ]:
runs_df.copy().sort_values(by='station_id')['station_id']

In [ ]:
runs_df_linreg.sort_values(by='station_id')['station_id']

In [ ]:
# Combine runs_df with runs_df_linreg by adding columns
df_combined = runs_df.copy().sort_values(by='station_id').reset_index(drop=True)

# Add columns from runs_df_linreg to df_combined
for column in runs_df_linreg.columns:
    if column not in df_combined.columns:
        df_combined[column] = runs_df_linreg.copy().sort_values(by='station_id').reset_index(drop=True)[column]
    else:
        # If column already exists, add with suffix
        df_combined[f"{column}_linreg"] = runs_df_linreg.copy().sort_values(by='station_id').reset_index(drop=True)[column]

# Filter each row to keep only common elements based on 'index' column
for i, row in df_combined.iterrows():
    # Get the dataframes
    arso_pred = row['arso_predictions_df']
    model_pred = row['model_predictions_df']
    actuals = row['actuals_df']
    arso_pred_linreg = row['arso_predictions_df_linreg']
    model_pred_linreg = row['model_predictions_df_linreg']
    actuals_linreg = row['actuals_df_linreg']
    
    # Convert index columns to datetime
    if arso_pred is not None and 'index' in arso_pred.columns:
        arso_pred['index'] = pd.to_datetime(arso_pred['index'], unit='ms')
    if model_pred is not None and 'index' in model_pred.columns:
        model_pred['index'] = pd.to_datetime(model_pred['index'], unit='ms')
    if actuals is not None and 'index' in actuals.columns:
        actuals['index'] = pd.to_datetime(actuals['index'], unit='ms')
    #if arso_pred_linreg is not None and 'index' in arso_pred_linreg.columns:
    #    arso_pred_linreg['index'] = arso_pred_linreg['index'].apply(lambda x: pd.Timestamp(pd.Timestamp(x).timestamp(), unit = 'us')).copy()
    if model_pred_linreg is not None and 'index' in model_pred_linreg.columns:
        model_pred_linreg['index'] = pd.to_datetime((model_pred_linreg['index']), unit='ms')
    if actuals_linreg is not None and 'index' in actuals_linreg.columns:
        actuals_linreg['index'] = pd.to_datetime((actuals_linreg['index']), unit='ms')
    arso_pred_linreg['index'] = actuals_linreg['index']
    
    # Check if all dataframes exist and have 'index' column
    dfs = [arso_pred, model_pred, actuals, arso_pred_linreg, model_pred_linreg, actuals_linreg]
    if all(df is not None and 'index' in df.columns for df in dfs):
        # Find common index values across all dataframes
        common_indices = set(arso_pred['index'])
        for df in dfs[1:]:
            common_indices = common_indices.intersection(set(df['index']))
        
        # Filter each dataframe to keep only rows with common indices
        df_combined.at[i, 'arso_predictions_df'] = arso_pred[arso_pred['index'].isin(common_indices)]
        df_combined.at[i, 'model_predictions_df'] = model_pred[model_pred['index'].isin(common_indices)]
        df_combined.at[i, 'actuals_df'] = actuals[actuals['index'].isin(common_indices)]
        df_combined.at[i, 'arso_predictions_df_linreg'] = arso_pred_linreg[arso_pred_linreg['index'].isin(common_indices)]
        df_combined.at[i, 'model_predictions_df_linreg'] = model_pred_linreg[model_pred_linreg['index'].isin(common_indices)]
        df_combined.at[i, 'actuals_df_linreg'] = actuals_linreg[actuals_linreg['index'].isin(common_indices)]
    else:
        print(f"No common indices found for station {row['station_id']}")
# Update runs_df to point to the combined dataframe
#df_combined



In [ ]:
df_combined['model_predictions_df'].iloc[10].shape, runs_df['model_predictions_df'].iloc[2].shape

In [ ]:
arso_pred_linreg['index']
#pd.to_datetime(runs_df['arso_predictions_df'].iloc[0]['index'], unit='ms'), pd.to_datetime(runs_df_linreg['arso_predictions_df_linreg'].iloc[0]['index'])#.shape

In [ ]:
df_combined.columns

In [ ]:
df_combined['station_id'] == df_combined['station_id_linreg']

In [ ]:
# MAP TIME TO DATETIME
# Loop over all rows in df_combined and convert 'index' column to datetime for each model_predictions_df
for i, row in df_combined.iterrows():
    modelpreddf = row['model_predictions_df']
    if modelpreddf is not None and 'index' in modelpreddf.columns:
        modelpreddf['index'] = pd.to_datetime(modelpreddf['index'], unit='ms')
        df_combined.at[i, 'model_predictions_df'] = modelpreddf


# read vv levels
vv_levels = pd.read_parquet('arso_vv.parquet')



for i, row in df_combined.iterrows():
    station_id = row['station_id']
    vv_levels_station = vv_levels[(vv_levels['station_id'].astype(int)) == station_id]

    
# For each run, check if actual Q in hours 4 onward surpasses any vv levels, and report the biggest vv passed (max vv3)
vv_report = []

# Group stations by first digit of station_id
station_groups = {}
for i, row in df_combined.iterrows():
    station_id = row['station_id']
    group_key = str(station_id)[0]  # First digit
    if group_key not in station_groups:
        station_groups[group_key] = []
    station_groups[group_key].append(i)

# Store results in a list for DataFrame creation
results_list = []

# Calculate VV statistics by group
for group_key, station_indices in station_groups.items():
    #print(f"\nGroup {group_key} (stations starting with {group_key}):")
    
    # Aggregate VV counts across all stations in this group
    group_vv_gt_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    group_vv_model_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    group_vv_arso_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    
    # Initialize confusion matrix statistics for each VV level
    # VV1 level (vv1+ vs no vv)
    model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn = 0, 0, 0, 0
    arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn = 0, 0, 0, 0
    
    # VV2 level (vv2+ vs vv1-)
    model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn = 0, 0, 0, 0
    arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn = 0, 0, 0, 0
    
    # VV3 level (vv3 vs vv2-)
    model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn = 0, 0, 0, 0
    arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn = 0, 0, 0, 0
    
    # Any VV level (any vv vs no vv)
    model_any_tp, model_any_tn, model_any_fp, model_any_fn = 0, 0, 0, 0
    arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn = 0, 0, 0, 0
    
    # TODO: TEST THIS
    HOUR_CUTOFF = 3 # first 3 hours should not be already vv
    END_HOUR = 18

    for i in station_indices:
        row = df_combined.iloc[i]
        actuals_df = row['actuals_df']
        station_id = row['station_id']
        vv_levels_station = vv_levels[vv_levels['station_id'].astype(int) == station_id]
        
        # Get vv1, vv2, vv3 for this station (if available)
        if len(vv_levels_station) == 0:
            continue
        vv1 = vv_levels_station['vv1'].values[0]
        vv2 = vv_levels_station['vv2'].values[0]
        vv3 = vv_levels_station['vv3'].values[0]

        rel = actuals_df[[str(i) for i in range(0, END_HOUR)]] # hourly predictions are 0...17
        rel_max = rel.max(axis=1)
        # For each row, assign the biggest vv level passed (vv3 > vv2 > vv1), or None if none passed
        def biggest_vv_passed(val):
            if val >= vv3:
                return 'vv3'
            elif val >= vv2:
                return 'vv2'
            elif val >= vv1:
                return 'vv1'
            else:
                return '-'
        biggest_vv_gt = rel_max.apply(biggest_vv_passed)
        rel_first6 = actuals_df[[str(i) for i in range(0, HOUR_CUTOFF)]] # hourly predictions are 0...17
        rel_first6_max = rel_first6.max(axis=1)
        biggest_vv_gt_first6 = rel_first6_max.apply(biggest_vv_passed).apply(lambda x: '???')
        print(f"biggest_vv_gt_first6: {biggest_vv_gt_first6}")

        # Only set vv value if there was no vv in the first 6 hours
        biggest_vv_gt = biggest_vv_gt.where(biggest_vv_gt_first6 != '-', '-')

        # now look at model predictions and determine the vv
        model_predictions_df = row['model_predictions_df']
        rel_model = model_predictions_df[[str(i) for i in range(00, END_HOUR)]]
        rel_model_max = rel_model.max(axis=1)
        biggest_vv_model = rel_model_max.apply(biggest_vv_passed)

        # Only set vv value if there was no vv in the first 6 hours
        biggest_vv_model = biggest_vv_model.where(biggest_vv_gt_first6 != '-', '-')
        
        # now look at arso predictions and determine the vv
        arso_predictions_df = row['arso_predictions_df']
        rel_arso = arso_predictions_df[[str(i) for i in range(0, END_HOUR)]]
        rel_arso_max = rel_arso.max(axis=1)
        biggest_vv_arso = rel_arso_max.apply(biggest_vv_passed)
        biggest_vv_arso = biggest_vv_arso.where(biggest_vv_gt_first6 != '-', '-')

        # Add counts to group totals
        vv_gt_counts = biggest_vv_gt.value_counts()
        vv_model_counts = biggest_vv_model.value_counts()
        vv_arso_counts = biggest_vv_arso.value_counts()
        
        for vv_level in ['-', 'vv1', 'vv2', 'vv3']:
            group_vv_gt_counts[vv_level] += vv_gt_counts.get(vv_level, 0)
            group_vv_model_counts[vv_level] += vv_model_counts.get(vv_level, 0)
            group_vv_arso_counts[vv_level] += vv_arso_counts.get(vv_level, 0)
        
        # Calculate confusion matrix statistics for this station per VV level
        
        # VV1+ (vv1, vv2, vv3) vs no VV (-)
        gt_has_vv1_plus = (biggest_vv_gt.isin(['vv1']))
        model_has_vv1_plus = (biggest_vv_model.isin(['vv1']))
        arso_has_vv1_plus = (biggest_vv_arso.isin(['vv1']))
        
        model_vv1_tp += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == True)).sum()
        model_vv1_tn += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == False)).sum()
        model_vv1_fp += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == True)).sum()
        model_vv1_fn += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == False)).sum()
        
        arso_vv1_tp += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == True)).sum()
        arso_vv1_tn += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == False)).sum()
        arso_vv1_fp += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == True)).sum()
        arso_vv1_fn += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == False)).sum()
        
        # VV2+ (vv2, vv3) vs VV1- (-, vv1)
        gt_has_vv2_plus = (biggest_vv_gt.isin(['vv2']))
        model_has_vv2_plus = (biggest_vv_model.isin(['vv2']))
        arso_has_vv2_plus = (biggest_vv_arso.isin(['vv2']))
        
        model_vv2_tp += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == True)).sum()
        model_vv2_tn += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == False)).sum()
        model_vv2_fp += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == True)).sum()
        model_vv2_fn += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == False)).sum()
        
        arso_vv2_tp += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == True)).sum()
        arso_vv2_tn += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == False)).sum()
        arso_vv2_fp += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == True)).sum()
        arso_vv2_fn += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == False)).sum()
        
        # VV3 vs VV2- (-, vv1, vv2)
        gt_has_vv3 = (biggest_vv_gt == 'vv3')
        model_has_vv3 = (biggest_vv_model == 'vv3')
        arso_has_vv3 = (biggest_vv_arso == 'vv3')
        
        model_vv3_tp += ((gt_has_vv3 == True) & (model_has_vv3 == True)).sum()
        model_vv3_tn += ((gt_has_vv3 == False) & (model_has_vv3 == False)).sum()
        model_vv3_fp += ((gt_has_vv3 == False) & (model_has_vv3 == True)).sum()
        model_vv3_fn += ((gt_has_vv3 == True) & (model_has_vv3 == False)).sum()
        
        arso_vv3_tp += ((gt_has_vv3 == True) & (arso_has_vv3 == True)).sum()
        arso_vv3_tn += ((gt_has_vv3 == False) & (arso_has_vv3 == False)).sum()
        arso_vv3_fp += ((gt_has_vv3 == False) & (arso_has_vv3 == True)).sum()
        arso_vv3_fn += ((gt_has_vv3 == True) & (arso_has_vv3 == False)).sum()
        
        # Any VV (vv1, vv2, vv3) vs no VV (-)
        gt_has_any_vv = (biggest_vv_gt != '-')
        model_has_any_vv = (biggest_vv_model != '-')
        arso_has_any_vv = (biggest_vv_arso != '-')
        
        model_any_tp += ((gt_has_any_vv == True) & (model_has_any_vv == True)).sum()
        model_any_tn += ((gt_has_any_vv == False) & (model_has_any_vv == False)).sum()
        model_any_fp += ((gt_has_any_vv == False) & (model_has_any_vv == True)).sum()
        model_any_fn += ((gt_has_any_vv == True) & (model_has_any_vv == False)).sum()
        
        arso_any_tp += ((gt_has_any_vv == True) & (arso_has_any_vv == True)).sum()
        arso_any_tn += ((gt_has_any_vv == False) & (arso_has_any_vv == False)).sum()
        arso_any_fp += ((gt_has_any_vv == False) & (arso_has_any_vv == True)).sum()
        arso_any_fn += ((gt_has_any_vv == True) & (arso_has_any_vv == False)).sum()
    
    # Calculate metrics for each VV level
    def calculate_metrics(tp, tn, fp, fn):
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
        return precision, recall, accuracy
    
    # VV1+ metrics
    model_vv1_precision, model_vv1_recall, model_vv1_accuracy = calculate_metrics(model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn)
    arso_vv1_precision, arso_vv1_recall, arso_vv1_accuracy = calculate_metrics(arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn)
    
    # VV2+ metrics
    model_vv2_precision, model_vv2_recall, model_vv2_accuracy = calculate_metrics(model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn)
    arso_vv2_precision, arso_vv2_recall, arso_vv2_accuracy = calculate_metrics(arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn)
    
    # VV3 metrics
    model_vv3_precision, model_vv3_recall, model_vv3_accuracy = calculate_metrics(model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn)
    arso_vv3_precision, arso_vv3_recall, arso_vv3_accuracy = calculate_metrics(arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn)
    
    # Any VV metrics
    model_any_precision, model_any_recall, model_any_accuracy = calculate_metrics(model_any_tp, model_any_tn, model_any_fp, model_any_fn)
    arso_any_precision, arso_any_recall, arso_any_accuracy = calculate_metrics(arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn)
    
    # Store results for this group
    results_list.append({
        'group': group_key,
        'gt_no_vv': group_vv_gt_counts['-'],
        'gt_vv1': group_vv_gt_counts['vv1'],
        'gt_vv2': group_vv_gt_counts['vv2'],
        'gt_vv3': group_vv_gt_counts['vv3'],
        'model_no_vv': group_vv_model_counts['-'],
        'model_vv1': group_vv_model_counts['vv1'],
        'model_vv2': group_vv_model_counts['vv2'],
        'model_vv3': group_vv_model_counts['vv3'],
        'arso_no_vv': group_vv_arso_counts['-'],
        'arso_vv1': group_vv_arso_counts['vv1'],
        'arso_vv2': group_vv_arso_counts['vv2'],
        'arso_vv3': group_vv_arso_counts['vv3'],
        # VV1+ metrics
        'model_vv1_tp': model_vv1_tp, 'model_vv1_tn': model_vv1_tn, 'model_vv1_fp': model_vv1_fp, 'model_vv1_fn': model_vv1_fn,
        'model_vv1_precision': model_vv1_precision, 'model_vv1_recall': model_vv1_recall, 'model_vv1_accuracy': model_vv1_accuracy,
        'arso_vv1_tp': arso_vv1_tp, 'arso_vv1_tn': arso_vv1_tn, 'arso_vv1_fp': arso_vv1_fp, 'arso_vv1_fn': arso_vv1_fn,
        'arso_vv1_precision': arso_vv1_precision, 'arso_vv1_recall': arso_vv1_recall, 'arso_vv1_accuracy': arso_vv1_accuracy,
        # VV2+ metrics
        'model_vv2_tp': model_vv2_tp, 'model_vv2_tn': model_vv2_tn, 'model_vv2_fp': model_vv2_fp, 'model_vv2_fn': model_vv2_fn,
        'model_vv2_precision': model_vv2_precision, 'model_vv2_recall': model_vv2_recall, 'model_vv2_accuracy': model_vv2_accuracy,
        'arso_vv2_tp': arso_vv2_tp, 'arso_vv2_tn': arso_vv2_tn, 'arso_vv2_fp': arso_vv2_fp, 'arso_vv2_fn': arso_vv2_fn,
        'arso_vv2_precision': arso_vv2_precision, 'arso_vv2_recall': arso_vv2_recall, 'arso_vv2_accuracy': arso_vv2_accuracy,
        # VV3 metrics
        'model_vv3_tp': model_vv3_tp, 'model_vv3_tn': model_vv3_tn, 'model_vv3_fp': model_vv3_fp, 'model_vv3_fn': model_vv3_fn,
        'model_vv3_precision': model_vv3_precision, 'model_vv3_recall': model_vv3_recall, 'model_vv3_accuracy': model_vv3_accuracy,
        'arso_vv3_tp': arso_vv3_tp, 'arso_vv3_tn': arso_vv3_tn, 'arso_vv3_fp': arso_vv3_fp, 'arso_vv3_fn': arso_vv3_fn,
        'arso_vv3_precision': arso_vv3_precision, 'arso_vv3_recall': arso_vv3_recall, 'arso_vv3_accuracy': arso_vv3_accuracy,
        # Any VV metrics
        'model_any_tp': model_any_tp, 'model_any_tn': model_any_tn, 'model_any_fp': model_any_fp, 'model_any_fn': model_any_fn,
        'model_any_precision': model_any_precision, 'model_any_recall': model_any_recall, 'model_any_accuracy': model_any_accuracy,
        'arso_any_tp': arso_any_tp, 'arso_any_tn': arso_any_tn, 'arso_any_fp': arso_any_fp, 'arso_any_fn': arso_any_fn,
        'arso_any_precision': arso_any_precision, 'arso_any_recall': arso_any_recall, 'arso_any_accuracy': arso_any_accuracy
    })

# Create DataFrame from results
vv_results_df = pd.DataFrame(results_list)

# Display the results table
print("VV Warning Performance by Station Group and VV Level:")
print("="*80)
display(vv_results_df)

# Create visualizations
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('VV Warning Performance Analysis by Station Group and VV Level', fontsize=16, fontweight='bold')

x_pos = np.arange(len(vv_results_df))
width = 0.35

# VV1+ Analysis
ax1 = axes[0, 0]
ax1.bar(x_pos - width/2, vv_results_df['model_vv1_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax1.bar(x_pos + width/2, vv_results_df['arso_vv1_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax1.set_xlabel('Station Group')
ax1.set_ylabel('Precision')
ax1.set_title('VV1+ Precision Comparison')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(vv_results_df['group'])
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.bar(x_pos - width/2, vv_results_df['model_vv1_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax2.bar(x_pos + width/2, vv_results_df['arso_vv1_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax2.set_xlabel('Station Group')
ax2.set_ylabel('Recall')
ax2.set_title('VV1+ Recall Comparison')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(vv_results_df['group'])
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
ax3.bar(x_pos - width/2, vv_results_df['model_vv1_accuracy'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax3.bar(x_pos + width/2, vv_results_df['arso_vv1_accuracy'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax3.set_xlabel('Station Group')
ax3.set_ylabel('Accuracy')
ax3.set_title('VV1+ Accuracy Comparison')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(vv_results_df['group'])
ax3.legend()
ax3.grid(True, alpha=0.3)

# VV2+ Analysis
ax4 = axes[1, 0]
ax4.bar(x_pos - width/2, vv_results_df['model_vv2_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax4.bar(x_pos + width/2, vv_results_df['arso_vv2_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax4.set_xlabel('Station Group')
ax4.set_ylabel('Precision')
ax4.set_title('VV2+ Precision Comparison')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(vv_results_df['group'])
ax4.legend()
ax4.grid(True, alpha=0.3)

ax5 = axes[1, 1]
ax5.bar(x_pos - width/2, vv_results_df['model_vv2_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax5.bar(x_pos + width/2, vv_results_df['arso_vv2_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax5.set_xlabel('Station Group')
ax5.set_ylabel('Recall')
ax5.set_title('VV2+ Recall Comparison')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(vv_results_df['group'])
ax5.legend()
ax5.grid(True, alpha=0.3)

ax6 = axes[1, 2]
ax6.bar(x_pos - width/2, vv_results_df['model_vv2_accuracy'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax6.bar(x_pos + width/2, vv_results_df['arso_vv2_accuracy'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax6.set_xlabel('Station Group')
ax6.set_ylabel('Accuracy')
ax6.set_title('VV2+ Accuracy Comparison')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(vv_results_df['group'])
ax6.legend()
ax6.grid(True, alpha=0.3)

# VV3 Analysis
ax7 = axes[2, 0]
ax7.bar(x_pos - width/2, vv_results_df['model_vv3_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax7.bar(x_pos + width/2, vv_results_df['arso_vv3_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax7.set_xlabel('Station Group')
ax7.set_ylabel('Precision')
ax7.set_title('VV3 Precision Comparison')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(vv_results_df['group'])
ax7.legend()
ax7.grid(True, alpha=0.3)

ax8 = axes[2, 1]
ax8.bar(x_pos - width/2, vv_results_df['model_vv3_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax8.bar(x_pos + width/2, vv_results_df['arso_vv3_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax8.set_xlabel('Station Group')
ax8.set_ylabel('Recall')
ax8.set_title('VV3 Recall Comparison')
ax8.set_xticks(x_pos)
ax8.set_xticklabels(vv_results_df['group'])
ax8.legend()
ax8.grid(True, alpha=0.3)

ax9 = axes[2, 2]
ax9.bar(x_pos - width/2, vv_results_df['model_vv3_accuracy'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax9.bar(x_pos + width/2, vv_results_df['arso_vv3_accuracy'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax9.set_xlabel('Station Group')
ax9.set_ylabel('Accuracy')
ax9.set_title('VV3 Accuracy Comparison')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(vv_results_df['group'])
ax9.legend()
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics per VV level
print("\nSummary Statistics by VV Level:")
print("-" * 60)
print("VV1+ (Any VV1 or higher):")
print(f"  Model Precision: {vv_results_df['model_vv1_precision'].mean():.3f} ± {vv_results_df['model_vv1_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv1_precision'].mean():.3f} ± {vv_results_df['arso_vv1_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv1_recall'].mean():.3f} ± {vv_results_df['model_vv1_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv1_recall'].mean():.3f} ± {vv_results_df['arso_vv1_recall'].std():.3f}")

print("\nVV2+ (VV2 or VV3):")
print(f"  Model Precision: {vv_results_df['model_vv2_precision'].mean():.3f} ± {vv_results_df['model_vv2_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv2_precision'].mean():.3f} ± {vv_results_df['arso_vv2_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv2_recall'].mean():.3f} ± {vv_results_df['model_vv2_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv2_recall'].mean():.3f} ± {vv_results_df['arso_vv2_recall'].std():.3f}")

print("\nVV3 (Highest level):")
print(f"  Model Precision: {vv_results_df['model_vv3_precision'].mean():.3f} ± {vv_results_df['model_vv3_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv3_precision'].mean():.3f} ± {vv_results_df['arso_vv3_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv3_recall'].mean():.3f} ± {vv_results_df['model_vv3_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv3_recall'].mean():.3f} ± {vv_results_df['arso_vv3_recall'].std():.3f}")

print("\nAny VV (Overall):")
print(f"  Model Precision: {vv_results_df['model_any_precision'].mean():.3f} ± {vv_results_df['model_any_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_any_precision'].mean():.3f} ± {vv_results_df['arso_any_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_any_recall'].mean():.3f} ± {vv_results_df['model_any_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_any_recall'].mean():.3f} ± {vv_results_df['arso_any_recall'].std():.3f}")

In [ ]:
def compute_vv_results(df_combined, vv_levels_path, pred_col='model_predictions_df'):
    """
    Compute VV statistics and metrics by station group, parametrized by prediction column name.

    Args:
        df_combined (pd.DataFrame): DataFrame containing station data.
        vv_levels_path (str): Path to the parquet file with vv levels.
        pred_col (str): Column name in df_combined for the model predictions DataFrame.

    Returns:
        pd.DataFrame: DataFrame with VV statistics and metrics by group.
    """
    import pandas as pd

    # MAP TIME TO DATETIME for the specified prediction column
    for i, row in df_combined.iterrows():
        pred_df = row[pred_col]
        if pred_df is not None and 'index' in pred_df.columns:
            pred_df.loc[:, 'index'] = pd.to_datetime(pred_df['index'], unit='ms')
            df_combined.at[i, pred_col] = pred_df

    # read vv levels
    vv_levels = pd.read_parquet(vv_levels_path)

    # Group stations by first digit of station_id
    station_groups = {}
    for i, row in df_combined.iterrows():
        station_id = row['station_id']
        group_key = str(station_id)[0]  # First digit
        if group_key not in station_groups:
            station_groups[group_key] = []
        station_groups[group_key].append(i)

    # Store results in a list for DataFrame creation
    results_list = []

    # Calculate VV statistics by group
    for group_key, station_indices in station_groups.items():
        # Aggregate VV counts across all stations in this group
        group_vv_gt_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
        group_vv_model_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
        group_vv_arso_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}

        # Initialize confusion matrix statistics for each VV level
        # VV1 level (vv1+ vs no vv)
        model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn = 0, 0, 0, 0
        arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn = 0, 0, 0, 0

        # VV2 level (vv2+ vs vv1-)
        model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn = 0, 0, 0, 0
        arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn = 0, 0, 0, 0

        # VV3 level (vv3 vs vv2-)
        model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn = 0, 0, 0, 0
        arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn = 0, 0, 0, 0

        # Any VV level (any vv vs no vv)
        model_any_tp, model_any_tn, model_any_fp, model_any_fn = 0, 0, 0, 0
        arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn = 0, 0, 0, 0

        HOUR_CUTOFF = 3 # first 3 hours should not be already vv
        print(f"There are {len(station_indices)} stations in group {group_key}")
        for i in station_indices:
            row = df_combined.iloc[i]
            actuals_df = row['actuals_df']
            station_id = row['station_id']
            upostevane_postaje = [1140, 1165, 1355, 1060, 2005, 2250, 2390, 2420, 2640, 2652, 2693, 2719, 2754, 2830, 2880, 2900, 3014, 3080, 3180,
                                   3200, 3250, 3260, 3300, 3420, 3465, 3530, 3660, 3725, 3850, 3900, 4025, 4050, 4155, 4200, 4209, 4222, 4230, 4270,
                                     4400, 4430, 4515, 4520, 4570, 4626, 4671, 4695, 4710, 4750, 4770, 4791, 4820, 4828, 4860, 4969, 5030, 5078, 5240,
                                       5330, 5425, 5479, 5540, 5770, 5840, 5940, 6060, 6068, 6140, 6200, 6210, 6240, 6280, 6300, 6340, 6415, 6550, 6630,
                                         6691, 6720, 6790, 6835, 7029, 7060, 7110, 7160, 7200, 7220, 7235, 7272, 7308, 7340, 7380, 8031, 8060, 8080, 8180,
                                           8270, 8332, 8350, 8450, 8454, 8478, 8500, 8561, 8565, 8591, 8601, 8610, 8630, 8660, 8680, 8700, 8710, 9015, 9030,
                                             9050, 9100, 9108, 9210, 9300]
            if station_id not in upostevane_postaje:
                print(f"Station {station_id} is not in the upostevane_postaje list")
                continue
            vv_levels_station = vv_levels[vv_levels['station_id'].astype(int) == station_id]

            # Get vv1, vv2, vv3 for this station (if available)
            if len(vv_levels_station) == 0:
                continue
            vv1 = vv_levels_station['vv1'].values[0]
            vv2 = vv_levels_station['vv2'].values[0]
            vv3 = vv_levels_station['vv3'].values[0]

            rel = actuals_df[[str(i) for i in range(HOUR_CUTOFF, 18)]] # hourly predictions are 0...17
            rel_max = rel.max(axis=1)
            # For each row, assign the biggest vv level passed (vv3 > vv2 > vv1), or None if none passed
            def biggest_vv_passed(val):
                if val >= vv3:
                    return 'vv3'
                elif val >= vv2:
                    return 'vv2'
                elif val >= vv1:
                    return 'vv1'
                else:
                    return '-'
            biggest_vv_gt = rel_max.apply(biggest_vv_passed)
            rel_first6 = actuals_df[[str(i) for i in range(0, HOUR_CUTOFF)]] # hourly predictions are 0...17
            rel_first6_max = rel_first6.max(axis=1)
            biggest_vv_gt_first6 = rel_first6_max.apply(biggest_vv_passed)

            # Only set vv value if there was no vv in the first 6 hours
            biggest_vv_gt = biggest_vv_gt.where(biggest_vv_gt_first6 != '-', '-')

            # now look at model predictions and determine the vv
            model_predictions_df = row[pred_col]
            rel_model = model_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
            rel_model_max = rel_model.max(axis=1)
            biggest_vv_model = rel_model_max.apply(biggest_vv_passed)

            # Only set vv value if there was no vv in the first 6 hours
            biggest_vv_model = biggest_vv_model.where(biggest_vv_gt_first6 != '-', '-')

            # now look at arso predictions and determine the vv
            arso_predictions_df = row['arso_predictions_df']
            rel_arso = arso_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
            rel_arso_max = rel_arso.max(axis=1)
            biggest_vv_arso = rel_arso_max.apply(biggest_vv_passed)
            biggest_vv_arso = biggest_vv_arso.where(biggest_vv_gt_first6 != '-', '-')

            # Add counts to group totals
            vv_gt_counts = biggest_vv_gt.value_counts()
            vv_model_counts = biggest_vv_model.value_counts()
            vv_arso_counts = biggest_vv_arso.value_counts()

            for vv_level in ['-', 'vv1', 'vv2', 'vv3']:
                group_vv_gt_counts[vv_level] += vv_gt_counts.get(vv_level, 0)
                group_vv_model_counts[vv_level] += vv_model_counts.get(vv_level, 0)
                group_vv_arso_counts[vv_level] += vv_arso_counts.get(vv_level, 0)

            # Calculate confusion matrix statistics for this station per VV level

            # VV1+ (vv1, vv2, vv3) vs no VV (-)
            gt_has_vv1_plus = (biggest_vv_gt.isin(['vv1']))
            model_has_vv1_plus = (biggest_vv_model.isin(['vv1']))
            arso_has_vv1_plus = (biggest_vv_arso.isin(['vv1']))

            model_vv1_tp += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == True)).sum()
            model_vv1_tn += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == False)).sum()
            model_vv1_fp += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == True)).sum()
            model_vv1_fn += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == False)).sum()

            arso_vv1_tp += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == True)).sum()
            arso_vv1_tn += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == False)).sum()
            arso_vv1_fp += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == True)).sum()
            arso_vv1_fn += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == False)).sum()

            # VV2+ (vv2, vv3) vs VV1- (-, vv1)
            gt_has_vv2_plus = (biggest_vv_gt.isin(['vv2']))
            model_has_vv2_plus = (biggest_vv_model.isin(['vv2']))
            arso_has_vv2_plus = (biggest_vv_arso.isin(['vv2']))

            model_vv2_tp += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == True)).sum()
            model_vv2_tn += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == False)).sum()
            model_vv2_fp += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == True)).sum()
            model_vv2_fn += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == False)).sum()

            arso_vv2_tp += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == True)).sum()
            arso_vv2_tn += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == False)).sum()
            arso_vv2_fp += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == True)).sum()
            arso_vv2_fn += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == False)).sum()

            # VV3 vs VV2- (-, vv1, vv2)
            gt_has_vv3 = (biggest_vv_gt == 'vv3')
            model_has_vv3 = (biggest_vv_model == 'vv3')
            arso_has_vv3 = (biggest_vv_arso == 'vv3')

            model_vv3_tp += ((gt_has_vv3 == True) & (model_has_vv3 == True)).sum()
            model_vv3_tn += ((gt_has_vv3 == False) & (model_has_vv3 == False)).sum()
            model_vv3_fp += ((gt_has_vv3 == False) & (model_has_vv3 == True)).sum()
            model_vv3_fn += ((gt_has_vv3 == True) & (model_has_vv3 == False)).sum()

            arso_vv3_tp += ((gt_has_vv3 == True) & (arso_has_vv3 == True)).sum()
            arso_vv3_tn += ((gt_has_vv3 == False) & (arso_has_vv3 == False)).sum()
            arso_vv3_fp += ((gt_has_vv3 == False) & (arso_has_vv3 == True)).sum()
            arso_vv3_fn += ((gt_has_vv3 == True) & (arso_has_vv3 == False)).sum()

            # Any VV (vv1, vv2, vv3) vs no VV (-)
            gt_has_any_vv = (biggest_vv_gt != '-')
            model_has_any_vv = (biggest_vv_model != '-')
            arso_has_any_vv = (biggest_vv_arso != '-')

            model_any_tp += ((gt_has_any_vv == True) & (model_has_any_vv == True)).sum()
            model_any_tn += ((gt_has_any_vv == False) & (model_has_any_vv == False)).sum()
            model_any_fp += ((gt_has_any_vv == False) & (model_has_any_vv == True)).sum()
            model_any_fn += ((gt_has_any_vv == True) & (model_has_any_vv == False)).sum()

            arso_any_tp += ((gt_has_any_vv == True) & (arso_has_any_vv == True)).sum()
            arso_any_tn += ((gt_has_any_vv == False) & (arso_has_any_vv == False)).sum()
            arso_any_fp += ((gt_has_any_vv == False) & (arso_has_any_vv == True)).sum()
            arso_any_fn += ((gt_has_any_vv == True) & (arso_has_any_vv == False)).sum()

        # Calculate metrics for each VV level
        def calculate_metrics(tp, tn, fp, fn):
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            return precision, recall

        # VV1+ metrics
        model_vv1_precision, model_vv1_recall = calculate_metrics(model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn)
        arso_vv1_precision, arso_vv1_recall = calculate_metrics(arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn)

        # VV2+ metrics
        model_vv2_precision, model_vv2_recall = calculate_metrics(model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn)
        arso_vv2_precision, arso_vv2_recall = calculate_metrics(arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn)

        # VV3 metrics
        model_vv3_precision, model_vv3_recall = calculate_metrics(model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn)
        arso_vv3_precision, arso_vv3_recall = calculate_metrics(arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn)

        # Any VV metrics
        model_any_precision, model_any_recall = calculate_metrics(model_any_tp, model_any_tn, model_any_fp, model_any_fn)
        arso_any_precision, arso_any_recall = calculate_metrics(arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn)

        # Store results for this group
        results_list.append({
            'group': group_key,
            'gt_no_vv': group_vv_gt_counts['-'],
            'gt_vv1': group_vv_gt_counts['vv1'],
            'gt_vv2': group_vv_gt_counts['vv2'],
            'gt_vv3': group_vv_gt_counts['vv3'],
            'model_no_vv': group_vv_model_counts['-'],
            'model_vv1': group_vv_model_counts['vv1'],
            'model_vv2': group_vv_model_counts['vv2'],
            'model_vv3': group_vv_model_counts['vv3'],
            'arso_no_vv': group_vv_arso_counts['-'],
            'arso_vv1': group_vv_arso_counts['vv1'],
            'arso_vv2': group_vv_arso_counts['vv2'],
            'arso_vv3': group_vv_arso_counts['vv3'],
            # VV1+ metrics
            'model_vv1_tp': model_vv1_tp, 'model_vv1_tn': model_vv1_tn, 'model_vv1_fp': model_vv1_fp, 'model_vv1_fn': model_vv1_fn,
            'model_vv1_precision': model_vv1_precision, 'model_vv1_recall': model_vv1_recall,
            'arso_vv1_tp': arso_vv1_tp, 'arso_vv1_tn': arso_vv1_tn, 'arso_vv1_fp': arso_vv1_fp, 'arso_vv1_fn': arso_vv1_fn,
            'arso_vv1_precision': arso_vv1_precision, 'arso_vv1_recall': arso_vv1_recall,
            # VV2+ metrics
            'model_vv2_tp': model_vv2_tp, 'model_vv2_tn': model_vv2_tn, 'model_vv2_fp': model_vv2_fp, 'model_vv2_fn': model_vv2_fn,
            'model_vv2_precision': model_vv2_precision, 'model_vv2_recall': model_vv2_recall,
            'arso_vv2_tp': arso_vv2_tp, 'arso_vv2_tn': arso_vv2_tn, 'arso_vv2_fp': arso_vv2_fp, 'arso_vv2_fn': arso_vv2_fn,
            'arso_vv2_precision': arso_vv2_precision, 'arso_vv2_recall': arso_vv2_recall,
            # VV3 metrics
            'model_vv3_tp': model_vv3_tp, 'model_vv3_tn': model_vv3_tn, 'model_vv3_fp': model_vv3_fp, 'model_vv3_fn': model_vv3_fn,
            'model_vv3_precision': model_vv3_precision, 'model_vv3_recall': model_vv3_recall,
            'arso_vv3_tp': arso_vv3_tp, 'arso_vv3_tn': arso_vv3_tn, 'arso_vv3_fp': arso_vv3_fp, 'arso_vv3_fn': arso_vv3_fn,
            'arso_vv3_precision': arso_vv3_precision, 'arso_vv3_recall': arso_vv3_recall,
            # Any VV metrics
            'model_any_tp': model_any_tp, 'model_any_tn': model_any_tn, 'model_any_fp': model_any_fp, 'model_any_fn': model_any_fn,
            'model_any_precision': model_any_precision, 'model_any_recall': model_any_recall,
            'arso_any_tp': arso_any_tp, 'arso_any_tn': arso_any_tn, 'arso_any_fp': arso_any_fp, 'arso_any_fn': arso_any_fn,
            'arso_any_precision': arso_any_precision, 'arso_any_recall': arso_any_recall
        })

    # Create DataFrame from results
    vv_results_df = pd.DataFrame(results_list)
    return vv_results_df

# Compute results for TOK model
vv_results_df_TOK = compute_vv_results(df_combined, 'arso_vv.parquet', pred_col='model_predictions_df')
vv_results_df_TOK = vv_results_df_TOK.rename(columns=lambda x: x.replace('model_', 'TOK_'))

# Compute results for linreg model
vv_results_df_linreg = compute_vv_results(df_combined, 'arso_vv.parquet', pred_col='model_predictions_df_linreg')
vv_results_df_linreg = vv_results_df_linreg.rename(columns=lambda x: x.replace('model_', 'linreg_'))

# Merge on 'group', 'gt_no_vv', 'gt_vv1', 'gt_vv2', 'gt_vv3', and arso columns (which should be the same)
merge_cols = ['group', 'gt_no_vv', 'gt_vv1', 'gt_vv2', 'gt_vv3', 
              'arso_no_vv', 'arso_vv1', 'arso_vv2', 'arso_vv3',
              'arso_vv1_tp', 'arso_vv1_tn', 'arso_vv1_fp', 'arso_vv1_fn',
              'arso_vv1_precision', 'arso_vv1_recall',
              'arso_vv2_tp', 'arso_vv2_tn', 'arso_vv2_fp', 'arso_vv2_fn',
              'arso_vv2_precision', 'arso_vv2_recall',
              'arso_vv3_tp', 'arso_vv3_tn', 'arso_vv3_fp', 'arso_vv3_fn',
              'arso_vv3_precision', 'arso_vv3_recall',
              'arso_any_tp', 'arso_any_tn', 'arso_any_fp', 'arso_any_fn',
              'arso_any_precision', 'arso_any_recall']

# Only keep one set of arso columns in the merged dataframe
vv_results_combined = pd.merge(
    vv_results_df_TOK, 
    vv_results_df_linreg, 
    on=merge_cols, 
    suffixes=('_TOK', '_linreg')
)
# Plotting for TOK, linreg, and ARSO

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 17,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 16,
    'figure.titlesize': 20
})

# Use vv_results_combined for plotting
df = vv_results_combined

x_pos = np.arange(len(df))
width = 0.25

# Define mapping from group number to descriptive label
group_labels_map = {
    1: "Pomurje",
    2: "Podravje",
    3: "Sava - glavni tok",
    4: "Pritoki Save",
    5: "Ljubljanica",
    6: "Savinja",
    7: "Krka",
    8: "Soča",
    9: "Jadranske reke"
}

# Prepare x axis labels in the order of df['group']
x_labels = [group_labels_map.get(int(g), str(g)) for g in df['group']]

fig, axes = plt.subplots(3, 3, figsize=(20, 20))
fig.suptitle('Analiza uspešnosti modelskih opozoril na posamezne visokovodne stopnje po skupinah postaj', fontsize=16, fontweight='bold')

legend_kwargs = dict(framealpha=0.3)  # More transparent legend

# --- 1. VVS Natancnost (Precision) ---
ax1 = axes[0, 0]
ax1.bar(x_pos - width, df['TOK_vv1_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax1.bar(x_pos, df['arso_vv1_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax1.bar(x_pos + width, df['linreg_vv1_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax1.set_xlabel('Skupina postaj')
ax1.set_ylabel('Natančnost')
ax1.set_title('Natančnost napovedovanja 1. VVS')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=20, ha='right')
ax1.set_ylim([0, 1])
ax1.legend(**legend_kwargs)
ax1.grid(True, alpha=0.3)

# --- 1. VVS Priklic (Recall) ---
ax2 = axes[0, 1]
ax2.bar(x_pos - width, df['TOK_vv1_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax2.bar(x_pos, df['arso_vv1_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax2.bar(x_pos + width, df['linreg_vv1_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax2.set_xlabel('Skupina postaj')
ax2.set_ylabel('Priklic')
ax2.set_title('Priklic napovedovanja 1. VVS')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_labels, rotation=20, ha='right')
ax2.set_ylim([0, 1])
ax2.legend(**legend_kwargs)
ax2.grid(True, alpha=0.3)

# --- 1. VVS GT Primeri ---
ax3 = axes[0, 2]
ax3.bar(x_pos, df['gt_vv1'], width, label='Primeri 1. VVS', alpha=0.8, color='green')
ax3.set_xlabel('Skupina postaj')
ax3.set_ylabel('Število primerov 1. VVS')
ax3.set_title('Število primerov 1. VVS')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(x_labels, rotation=20, ha='right')
ax3.legend(**legend_kwargs, loc='upper left')
ax3.grid(True, alpha=0.3)

# --- 2. VVS Natancnost (Precision) ---
ax4 = axes[1, 0]
ax4.bar(x_pos - width, df['TOK_vv2_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax4.bar(x_pos, df['arso_vv2_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax4.bar(x_pos + width, df['linreg_vv2_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax4.set_xlabel('Skupina postaj')
ax4.set_ylabel('Natančnost')
ax4.set_title('Natančnost napovedovanja 2. VVS')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(x_labels, rotation=20, ha='right')
ax4.set_ylim([0, 1])
ax4.legend(**legend_kwargs)
ax4.grid(True, alpha=0.3)

# --- 2. VVS Priklic (Recall) ---
ax5 = axes[1, 1]
ax5.bar(x_pos - width, df['TOK_vv2_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax5.bar(x_pos, df['arso_vv2_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax5.bar(x_pos + width, df['linreg_vv2_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax5.set_xlabel('Skupina postaj')
ax5.set_ylabel('Priklic')
ax5.set_title('Priklic napovedovanja 2. VVS')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(x_labels, rotation=20, ha='right')
ax5.set_ylim([0, 1])
ax5.legend(**legend_kwargs)
ax5.grid(True, alpha=0.3)

# --- 2. VVS GT Primeri ---
ax6 = axes[1, 2]
ax6.bar(x_pos, df['gt_vv2'], width, label='Primeri 2. VVS', alpha=0.8, color='orange')
ax6.set_xlabel('Skupina postaj')
ax6.set_ylabel('Število primerov 2. VVS')
ax6.set_title('Število primerov 2. VVS')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(x_labels, rotation=20, ha='right')
ax6.legend(**legend_kwargs, loc='upper right')
ax6.grid(True, alpha=0.3)

# --- 3. VVS Natancnost (Precision) ---
ax7 = axes[2, 0]
ax7.bar(x_pos - width, df['TOK_vv3_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax7.bar(x_pos, df['arso_vv3_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax7.bar(x_pos + width, df['linreg_vv3_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax7.set_xlabel('Skupina postaj')
ax7.set_ylabel('Natančnost')
ax7.set_title('Natančnost napovedovanja 3. VVS')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(x_labels, rotation=20, ha='right')
ax7.set_ylim([0, 1])
ax7.legend(**legend_kwargs)
ax7.grid(True, alpha=0.3)

# --- 3. VVS Priklic (Recall) ---
ax8 = axes[2, 1]
ax8.bar(x_pos - width, df['TOK_vv3_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax8.bar(x_pos, df['arso_vv3_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax8.bar(x_pos + width, df['linreg_vv3_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax8.set_xlabel('Skupina postaj')
ax8.set_ylabel('Priklic')
ax8.set_title('Priklic napovedovanja 3. VVS')
ax8.set_xticks(x_pos)
ax8.set_xticklabels(x_labels, rotation=20, ha='right')
ax8.set_ylim([0, 1])
ax8.legend(**legend_kwargs)
ax8.grid(True, alpha=0.3)

# --- 3. VVS GT Primeri ---
ax9 = axes[2, 2]
ax9.bar(x_pos, df['gt_vv3'], width, label='Primeri 3. VVS', alpha=0.8, color='red')
ax9.set_xlabel('Skupina postaj')
ax9.set_ylabel('Število primerov 3. VVS')
ax9.set_title('Število primerov 3. VVS')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(x_labels, rotation=20, ha='right')
ax9.legend(**legend_kwargs, loc='upper right')
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 17,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 16,
    'figure.titlesize': 20
})

# Use vv_results_combined for plotting
df = vv_results_combined

x_pos = np.arange(len(df))
width = 0.25

# Define mapping from group number to descriptive label
group_labels_map = {
    1: "Pomurje",
    2: "Podravje",
    3: "Sava - glavni tok",
    4: "Pritoki Save",
    5: "Ljubljanica",
    6: "Savinja",
    7: "Krka",
    8: "Soča",
    9: "Jadranske reke"
}

# Prepare x axis labels in the order of df['group']
x_labels = [group_labels_map.get(int(g), str(g)) for g in df['group']]

fig, axes = plt.subplots(4, 2, figsize=(16, 24))
fig.suptitle('Analiza uspešnosti modelskih opozoril na posamezne visokovodne stopnje po skupinah postaj', fontsize=18, fontweight='bold')

legend_kwargs = dict(framealpha=0.3)  # More transparent legend

# --- 1. VVS Natancnost (Precision) ---
ax1 = axes[0, 0]
ax1.bar(x_pos - width, df['TOK_vv1_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax1.bar(x_pos, df['arso_vv1_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax1.bar(x_pos + width, df['linreg_vv1_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax1.set_xlabel('Skupina postaj')
ax1.set_ylabel('Natančnost')
ax1.set_title('Natančnost napovedovanja 1. VVS')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=20, ha='right')
ax1.set_ylim([0, 1])
ax1.legend(**legend_kwargs)
ax1.grid(True, alpha=0.3)

# --- 1. VVS Priklic (Recall) ---
ax2 = axes[0, 1]
ax2.bar(x_pos - width, df['TOK_vv1_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax2.bar(x_pos, df['arso_vv1_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax2.bar(x_pos + width, df['linreg_vv1_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax2.set_xlabel('Skupina postaj')
ax2.set_ylabel('Priklic')
ax2.set_title('Priklic napovedovanja 1. VVS')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_labels, rotation=20, ha='right')
ax2.set_ylim([0, 1])
ax2.legend(**legend_kwargs)
ax2.grid(True, alpha=0.3)

# --- 2. VVS Natancnost (Precision) ---
ax3 = axes[1, 0]
ax3.bar(x_pos - width, df['TOK_vv2_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax3.bar(x_pos, df['arso_vv2_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax3.bar(x_pos + width, df['linreg_vv2_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax3.set_xlabel('Skupina postaj')
ax3.set_ylabel('Natančnost')
ax3.set_title('Natančnost napovedovanja 2. VVS')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(x_labels, rotation=20, ha='right')
ax3.set_ylim([0, 1])
ax3.legend(**legend_kwargs)
ax3.grid(True, alpha=0.3)

# --- 2. VVS Priklic (Recall) ---
ax4 = axes[1, 1]
ax4.bar(x_pos - width, df['TOK_vv2_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax4.bar(x_pos, df['arso_vv2_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax4.bar(x_pos + width, df['linreg_vv2_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax4.set_xlabel('Skupina postaj')
ax4.set_ylabel('Priklic')
ax4.set_title('Priklic napovedovanja 2. VVS')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(x_labels, rotation=20, ha='right')
ax4.set_ylim([0, 1])
ax4.legend(**legend_kwargs)
ax4.grid(True, alpha=0.3)

# --- 3. VVS Natancnost (Precision) ---
ax5 = axes[2, 0]
ax5.bar(x_pos - width, df['TOK_vv3_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax5.bar(x_pos, df['arso_vv3_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax5.bar(x_pos + width, df['linreg_vv3_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax5.set_xlabel('Skupina postaj')
ax5.set_ylabel('Natančnost')
ax5.set_title('Natančnost napovedovanja 3. VVS')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(x_labels, rotation=20, ha='right')
ax5.set_ylim([0, 1])
ax5.legend(**legend_kwargs)
ax5.grid(True, alpha=0.3)

# --- 3. VVS Priklic (Recall) ---
ax6 = axes[2, 1]
ax6.bar(x_pos - width, df['TOK_vv3_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax6.bar(x_pos, df['arso_vv3_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax6.bar(x_pos + width, df['linreg_vv3_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax6.set_xlabel('Skupina postaj')
ax6.set_ylabel('Priklic')
ax6.set_title('Priklic napovedovanja 3. VVS')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(x_labels, rotation=20, ha='right')
ax6.set_ylim([0, 1])
ax6.legend(**legend_kwargs)
ax6.grid(True, alpha=0.3)

# --- Število primerov po VVS (side by side, not stacked) ---
bar_width = width  # reuse width for consistency
offset = bar_width

# Levo: samo 1. VVS
ax7 = axes[3, 0]
ax7.bar(x_pos, df['gt_vv1'], bar_width, label='Primeri 1. VVS', alpha=0.8, color='skyblue')
ax7.set_xlabel('Skupina postaj')
ax7.set_ylabel('Število primerov')
ax7.set_title('Število primerov 1. VVS')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(x_labels, rotation=20, ha='right')
ax7.legend(**legend_kwargs, loc='upper left')
ax7.grid(True, alpha=0.3)

# Desno: 2. in 3. VVS - side by side, ne skupaj
ax8 = axes[3, 1]
bar_width = width
offset = bar_width / 2

ax8.bar(x_pos - offset, df['gt_vv2'], bar_width, label='Primeri 2. VVS', alpha=0.8, color='lightcoral')
ax8.bar(x_pos + offset, df['gt_vv3'], bar_width, label='Primeri 3. VVS', alpha=0.8, color='orange')
ax8.set_xlabel('Skupina postaj')
ax8.set_ylabel('Število primerov')
ax8.set_title('Število primerov 2. in 3. VVS')
ax8.set_xticks(x_pos)
ax8.set_xticklabels(x_labels, rotation=20, ha='right')
ax8.legend(**legend_kwargs, loc='upper left')
ax8.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

import os
# Save the figure to a file
filename = 'rezultatiVVS.png'
filepath = os.path.join(FIGURES_DIR, filename)

# Save the figure
fig.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Graf shranjen v: {filepath}")



In [ ]:
######################## aggregirano za vse postaje 
def compute_vv_results(df_combined, vv_levels_path, pred_col='model_predictions_df'):
    """
    Compute VV statistics and metrics by station group, parametrized by prediction column name.

    Args:
        df_combined (pd.DataFrame): DataFrame containing station data.
        vv_levels_path (str): Path to the parquet file with vv levels.
        pred_col (str): Column name in df_combined for the model predictions DataFrame.

    Returns:
        pd.DataFrame: DataFrame with VV statistics and metrics by group.
    """
    import pandas as pd

    # MAP TIME TO DATETIME for the specified prediction column
    for i, row in df_combined.iterrows():
        pred_df = row[pred_col]
        if pred_df is not None and 'index' in pred_df.columns:
            pred_df.loc[:, 'index'] = pd.to_datetime(pred_df['index'], unit='ms')
            df_combined.at[i, pred_col] = pred_df

    # read vv levels
    vv_levels = pd.read_parquet(vv_levels_path)

    # Group stations by first digit of station_id
    station_groups = {}
    for i, row in df_combined.iterrows():
        station_id = 1
        group_key = str(station_id)[0]  # First digit
        if group_key not in station_groups:
            station_groups[group_key] = []
        station_groups[group_key].append(i)

    # Store results in a list for DataFrame creation
    results_list = []

    # Calculate VV statistics by group
    for group_key, station_indices in station_groups.items():
        # Aggregate VV counts across all stations in this group
        group_vv_gt_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
        group_vv_model_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
        group_vv_arso_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}

        # Initialize confusion matrix statistics for each VV level
        # VV1 level (vv1+ vs no vv)
        model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn = 0, 0, 0, 0
        arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn = 0, 0, 0, 0

        # VV2 level (vv2+ vs vv1-)
        model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn = 0, 0, 0, 0
        arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn = 0, 0, 0, 0

        # VV3 level (vv3 vs vv2-)
        model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn = 0, 0, 0, 0
        arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn = 0, 0, 0, 0

        # Any VV level (any vv vs no vv)
        model_any_tp, model_any_tn, model_any_fp, model_any_fn = 0, 0, 0, 0
        arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn = 0, 0, 0, 0

        HOUR_CUTOFF = 3 # first 3 hours should not be already vv
        print(f"There are {len(station_indices)} stations in group {group_key}")
        for i in station_indices:
            row = df_combined.iloc[i]
            actuals_df = row['actuals_df']
            station_id = row['station_id']
            upostevane_postaje = [1140, 1165, 1355, 1060, 2005, 2250, 2390, 2420, 2640, 2652, 2693, 2719, 2754, 2830, 2880, 2900, 3014, 3080, 3180,
                                   3200, 3250, 3260, 3300, 3420, 3465, 3530, 3660, 3725, 3850, 3900, 4025, 4050, 4155, 4200, 4209, 4222, 4230, 4270,
                                     4400, 4430, 4515, 4520, 4570, 4626, 4671, 4695, 4710, 4750, 4770, 4791, 4820, 4828, 4860, 4969, 5030, 5078, 5240,
                                       5330, 5425, 5479, 5540, 5770, 5840, 5940, 6060, 6068, 6140, 6200, 6210, 6240, 6280, 6300, 6340, 6415, 6550, 6630,
                                         6691, 6720, 6790, 6835, 7029, 7060, 7110, 7160, 7200, 7220, 7235, 7272, 7308, 7340, 7380, 8031, 8060, 8080, 8180,
                                           8270, 8332, 8350, 8450, 8454, 8478, 8500, 8561, 8565, 8591, 8601, 8610, 8630, 8660, 8680, 8700, 8710, 9015, 9030,
                                             9050, 9100, 9108, 9210, 9300]
            if station_id not in upostevane_postaje:
                print(f"Station {station_id} is not in the upostevane_postaje list")
                continue
            vv_levels_station = vv_levels[vv_levels['station_id'].astype(int) == station_id]

            # Get vv1, vv2, vv3 for this station (if available)
            if len(vv_levels_station) == 0:
                continue
            vv1 = vv_levels_station['vv1'].values[0]
            vv2 = vv_levels_station['vv2'].values[0]
            vv3 = vv_levels_station['vv3'].values[0]

            rel = actuals_df[[str(i) for i in range(HOUR_CUTOFF, 18)]] # hourly predictions are 0...17
            rel_max = rel.max(axis=1)
            # For each row, assign the biggest vv level passed (vv3 > vv2 > vv1), or None if none passed
            def biggest_vv_passed(val):
                if val >= vv3:
                    return 'vv3'
                elif val >= vv2:
                    return 'vv2'
                elif val >= vv1:
                    return 'vv1'
                else:
                    return '-'
            biggest_vv_gt = rel_max.apply(biggest_vv_passed)
            rel_first6 = actuals_df[[str(i) for i in range(0, HOUR_CUTOFF)]] # hourly predictions are 0...17
            rel_first6_max = rel_first6.max(axis=1)
            biggest_vv_gt_first6 = rel_first6_max.apply(biggest_vv_passed)

            # Only set vv value if there was no vv in the first 6 hours
            biggest_vv_gt = biggest_vv_gt.where(biggest_vv_gt_first6 != '-', '-')

            # now look at model predictions and determine the vv
            model_predictions_df = row[pred_col]
            rel_model = model_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
            rel_model_max = rel_model.max(axis=1)
            biggest_vv_model = rel_model_max.apply(biggest_vv_passed)

            # Only set vv value if there was no vv in the first 6 hours
            biggest_vv_model = biggest_vv_model.where(biggest_vv_gt_first6 != '-', '-')

            # now look at arso predictions and determine the vv
            arso_predictions_df = row['arso_predictions_df']
            rel_arso = arso_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
            rel_arso_max = rel_arso.max(axis=1)
            biggest_vv_arso = rel_arso_max.apply(biggest_vv_passed)
            biggest_vv_arso = biggest_vv_arso.where(biggest_vv_gt_first6 != '-', '-')

            # Add counts to group totals
            vv_gt_counts = biggest_vv_gt.value_counts()
            vv_model_counts = biggest_vv_model.value_counts()
            vv_arso_counts = biggest_vv_arso.value_counts()

            for vv_level in ['-', 'vv1', 'vv2', 'vv3']:
                group_vv_gt_counts[vv_level] += vv_gt_counts.get(vv_level, 0)
                group_vv_model_counts[vv_level] += vv_model_counts.get(vv_level, 0)
                group_vv_arso_counts[vv_level] += vv_arso_counts.get(vv_level, 0)

            # Calculate confusion matrix statistics for this station per VV level

            # VV1+ (vv1, vv2, vv3) vs no VV (-)
            gt_has_vv1_plus = (biggest_vv_gt.isin(['vv1']))
            model_has_vv1_plus = (biggest_vv_model.isin(['vv1']))
            arso_has_vv1_plus = (biggest_vv_arso.isin(['vv1']))

            model_vv1_tp += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == True)).sum()
            model_vv1_tn += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == False)).sum()
            model_vv1_fp += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == True)).sum()
            model_vv1_fn += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == False)).sum()

            arso_vv1_tp += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == True)).sum()
            arso_vv1_tn += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == False)).sum()
            arso_vv1_fp += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == True)).sum()
            arso_vv1_fn += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == False)).sum()

            # VV2+ (vv2, vv3) vs VV1- (-, vv1)
            gt_has_vv2_plus = (biggest_vv_gt.isin(['vv2']))
            model_has_vv2_plus = (biggest_vv_model.isin(['vv2']))
            arso_has_vv2_plus = (biggest_vv_arso.isin(['vv2']))

            model_vv2_tp += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == True)).sum()
            model_vv2_tn += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == False)).sum()
            model_vv2_fp += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == True)).sum()
            model_vv2_fn += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == False)).sum()

            arso_vv2_tp += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == True)).sum()
            arso_vv2_tn += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == False)).sum()
            arso_vv2_fp += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == True)).sum()
            arso_vv2_fn += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == False)).sum()

            # VV3 vs VV2- (-, vv1, vv2)
            gt_has_vv3 = (biggest_vv_gt == 'vv3')
            model_has_vv3 = (biggest_vv_model == 'vv3')
            arso_has_vv3 = (biggest_vv_arso == 'vv3')

            model_vv3_tp += ((gt_has_vv3 == True) & (model_has_vv3 == True)).sum()
            model_vv3_tn += ((gt_has_vv3 == False) & (model_has_vv3 == False)).sum()
            model_vv3_fp += ((gt_has_vv3 == False) & (model_has_vv3 == True)).sum()
            model_vv3_fn += ((gt_has_vv3 == True) & (model_has_vv3 == False)).sum()

            arso_vv3_tp += ((gt_has_vv3 == True) & (arso_has_vv3 == True)).sum()
            arso_vv3_tn += ((gt_has_vv3 == False) & (arso_has_vv3 == False)).sum()
            arso_vv3_fp += ((gt_has_vv3 == False) & (arso_has_vv3 == True)).sum()
            arso_vv3_fn += ((gt_has_vv3 == True) & (arso_has_vv3 == False)).sum()

            # Any VV (vv1, vv2, vv3) vs no VV (-)
            gt_has_any_vv = (biggest_vv_gt != '-')
            model_has_any_vv = (biggest_vv_model != '-')
            arso_has_any_vv = (biggest_vv_arso != '-')

            model_any_tp += ((gt_has_any_vv == True) & (model_has_any_vv == True)).sum()
            model_any_tn += ((gt_has_any_vv == False) & (model_has_any_vv == False)).sum()
            model_any_fp += ((gt_has_any_vv == False) & (model_has_any_vv == True)).sum()
            model_any_fn += ((gt_has_any_vv == True) & (model_has_any_vv == False)).sum()

            arso_any_tp += ((gt_has_any_vv == True) & (arso_has_any_vv == True)).sum()
            arso_any_tn += ((gt_has_any_vv == False) & (arso_has_any_vv == False)).sum()
            arso_any_fp += ((gt_has_any_vv == False) & (arso_has_any_vv == True)).sum()
            arso_any_fn += ((gt_has_any_vv == True) & (arso_has_any_vv == False)).sum()

        # Calculate metrics for each VV level
        def calculate_metrics(tp, tn, fp, fn):
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            return precision, recall

        # VV1+ metrics
        model_vv1_precision, model_vv1_recall = calculate_metrics(model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn)
        arso_vv1_precision, arso_vv1_recall = calculate_metrics(arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn)

        # VV2+ metrics
        model_vv2_precision, model_vv2_recall = calculate_metrics(model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn)
        arso_vv2_precision, arso_vv2_recall = calculate_metrics(arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn)

        # VV3 metrics
        model_vv3_precision, model_vv3_recall = calculate_metrics(model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn)
        arso_vv3_precision, arso_vv3_recall = calculate_metrics(arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn)

        # Any VV metrics
        model_any_precision, model_any_recall = calculate_metrics(model_any_tp, model_any_tn, model_any_fp, model_any_fn)
        arso_any_precision, arso_any_recall = calculate_metrics(arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn)

        # Store results for this group
        results_list.append({
            'group': group_key,
            'gt_no_vv': group_vv_gt_counts['-'],
            'gt_vv1': group_vv_gt_counts['vv1'],
            'gt_vv2': group_vv_gt_counts['vv2'],
            'gt_vv3': group_vv_gt_counts['vv3'],
            'model_no_vv': group_vv_model_counts['-'],
            'model_vv1': group_vv_model_counts['vv1'],
            'model_vv2': group_vv_model_counts['vv2'],
            'model_vv3': group_vv_model_counts['vv3'],
            'arso_no_vv': group_vv_arso_counts['-'],
            'arso_vv1': group_vv_arso_counts['vv1'],
            'arso_vv2': group_vv_arso_counts['vv2'],
            'arso_vv3': group_vv_arso_counts['vv3'],
            # VV1+ metrics
            'model_vv1_tp': model_vv1_tp, 'model_vv1_tn': model_vv1_tn, 'model_vv1_fp': model_vv1_fp, 'model_vv1_fn': model_vv1_fn,
            'model_vv1_precision': model_vv1_precision, 'model_vv1_recall': model_vv1_recall,
            'arso_vv1_tp': arso_vv1_tp, 'arso_vv1_tn': arso_vv1_tn, 'arso_vv1_fp': arso_vv1_fp, 'arso_vv1_fn': arso_vv1_fn,
            'arso_vv1_precision': arso_vv1_precision, 'arso_vv1_recall': arso_vv1_recall,
            # VV2+ metrics
            'model_vv2_tp': model_vv2_tp, 'model_vv2_tn': model_vv2_tn, 'model_vv2_fp': model_vv2_fp, 'model_vv2_fn': model_vv2_fn,
            'model_vv2_precision': model_vv2_precision, 'model_vv2_recall': model_vv2_recall,
            'arso_vv2_tp': arso_vv2_tp, 'arso_vv2_tn': arso_vv2_tn, 'arso_vv2_fp': arso_vv2_fp, 'arso_vv2_fn': arso_vv2_fn,
            'arso_vv2_precision': arso_vv2_precision, 'arso_vv2_recall': arso_vv2_recall,
            # VV3 metrics
            'model_vv3_tp': model_vv3_tp, 'model_vv3_tn': model_vv3_tn, 'model_vv3_fp': model_vv3_fp, 'model_vv3_fn': model_vv3_fn,
            'model_vv3_precision': model_vv3_precision, 'model_vv3_recall': model_vv3_recall,
            'arso_vv3_tp': arso_vv3_tp, 'arso_vv3_tn': arso_vv3_tn, 'arso_vv3_fp': arso_vv3_fp, 'arso_vv3_fn': arso_vv3_fn,
            'arso_vv3_precision': arso_vv3_precision, 'arso_vv3_recall': arso_vv3_recall,
            # Any VV metrics
            'model_any_tp': model_any_tp, 'model_any_tn': model_any_tn, 'model_any_fp': model_any_fp, 'model_any_fn': model_any_fn,
            'model_any_precision': model_any_precision, 'model_any_recall': model_any_recall,
            'arso_any_tp': arso_any_tp, 'arso_any_tn': arso_any_tn, 'arso_any_fp': arso_any_fp, 'arso_any_fn': arso_any_fn,
            'arso_any_precision': arso_any_precision, 'arso_any_recall': arso_any_recall
        })

    # Create DataFrame from results
    vv_results_df = pd.DataFrame(results_list)
    return vv_results_df

# Compute results for TOK model
vv_results_df_TOK = compute_vv_results(df_combined, 'arso_vv.parquet', pred_col='model_predictions_df')
vv_results_df_TOK = vv_results_df_TOK.rename(columns=lambda x: x.replace('model_', 'TOK_'))

# Compute results for linreg model
vv_results_df_linreg = compute_vv_results(df_combined, 'arso_vv.parquet', pred_col='model_predictions_df_linreg')
vv_results_df_linreg = vv_results_df_linreg.rename(columns=lambda x: x.replace('model_', 'linreg_'))

# Merge on 'group', 'gt_no_vv', 'gt_vv1', 'gt_vv2', 'gt_vv3', and arso columns (which should be the same)
merge_cols = ['group', 'gt_no_vv', 'gt_vv1', 'gt_vv2', 'gt_vv3', 
              'arso_no_vv', 'arso_vv1', 'arso_vv2', 'arso_vv3',
              'arso_vv1_tp', 'arso_vv1_tn', 'arso_vv1_fp', 'arso_vv1_fn',
              'arso_vv1_precision', 'arso_vv1_recall',
              'arso_vv2_tp', 'arso_vv2_tn', 'arso_vv2_fp', 'arso_vv2_fn',
              'arso_vv2_precision', 'arso_vv2_recall',
              'arso_vv3_tp', 'arso_vv3_tn', 'arso_vv3_fp', 'arso_vv3_fn',
              'arso_vv3_precision', 'arso_vv3_recall',
              'arso_any_tp', 'arso_any_tn', 'arso_any_fp', 'arso_any_fn',
              'arso_any_precision', 'arso_any_recall']

# Only keep one set of arso columns in the merged dataframe
vv_results_combined = pd.merge(
    vv_results_df_TOK, 
    vv_results_df_linreg, 
    on=merge_cols, 
    suffixes=('_TOK', '_linreg')
)
# Plotting for TOK, linreg, and ARSO

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 17,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 16,
    'figure.titlesize': 20
})

# Use vv_results_combined for plotting
df = vv_results_combined

x_pos = np.arange(len(df))
width = 0.25

# Define mapping from group number to descriptive label
group_labels_map = {
    1: "Vse postaje",
}

# Prepare x axis labels in the order of df['group']
x_labels = [group_labels_map.get(int(g), str(g)) for g in df['group']]

fig, axes = plt.subplots(3, 3, figsize=(20, 20))
fig.suptitle('Analiza uspešnosti modelskih opozoril na posamezne visokovodne stopnje po skupinah postaj', fontsize=16, fontweight='bold')

legend_kwargs = dict(framealpha=0.3)  # More transparent legend
print(df['TOK_vv1_precision'])
# --- 1. VVS Natancnost (Precision) ---
ax1 = axes[0, 0]
ax1.bar(x_pos - width, df['TOK_vv1_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax1.bar(x_pos, df['arso_vv1_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax1.bar(x_pos + width, df['linreg_vv1_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax1.set_xlabel('Skupina postaj')
ax1.set_ylabel('Natančnost')
ax1.set_title('Natančnost napovedovanja 1. VVS')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=20, ha='right')
ax1.set_ylim([0, 1])
ax1.legend(**legend_kwargs)
ax1.grid(True, alpha=0.3)

# --- 1. VVS Priklic (Recall) ---
ax2 = axes[0, 1]
ax2.bar(x_pos - width, df['TOK_vv1_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax2.bar(x_pos, df['arso_vv1_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax2.bar(x_pos + width, df['linreg_vv1_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax2.set_xlabel('Skupina postaj')
ax2.set_ylabel('Priklic')
ax2.set_title('Priklic napovedovanja 1. VVS')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(x_labels, rotation=20, ha='right')
ax2.set_ylim([0, 1])
ax2.legend(**legend_kwargs)
ax2.grid(True, alpha=0.3)

# --- 1. VVS GT Primeri ---
ax3 = axes[0, 2]
ax3.bar(x_pos, df['gt_vv1'], width, label='Primeri 1. VVS', alpha=0.8, color='green')
ax3.set_xlabel('Skupina postaj')
ax3.set_ylabel('Število primerov 1. VVS')
ax3.set_title('Število primerov 1. VVS')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(x_labels, rotation=20, ha='right')
ax3.legend(**legend_kwargs, loc='upper left')
ax3.grid(True, alpha=0.3)

# --- 2. VVS Natancnost (Precision) ---
ax4 = axes[1, 0]
ax4.bar(x_pos - width, df['TOK_vv2_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax4.bar(x_pos, df['arso_vv2_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax4.bar(x_pos + width, df['linreg_vv2_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax4.set_xlabel('Skupina postaj')
ax4.set_ylabel('Natančnost')
ax4.set_title('Natančnost napovedovanja 2. VVS')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(x_labels, rotation=20, ha='right')
ax4.set_ylim([0, 1])
ax4.legend(**legend_kwargs)
ax4.grid(True, alpha=0.3)

# --- 2. VVS Priklic (Recall) ---
ax5 = axes[1, 1]
ax5.bar(x_pos - width, df['TOK_vv2_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax5.bar(x_pos, df['arso_vv2_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax5.bar(x_pos + width, df['linreg_vv2_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax5.set_xlabel('Skupina postaj')
ax5.set_ylabel('Priklic')
ax5.set_title('Priklic napovedovanja 2. VVS')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(x_labels, rotation=20, ha='right')
ax5.set_ylim([0, 1])
ax5.legend(**legend_kwargs)
ax5.grid(True, alpha=0.3)

# --- 2. VVS GT Primeri ---
ax6 = axes[1, 2]
ax6.bar(x_pos, df['gt_vv2'], width, label='Primeri 2. VVS', alpha=0.8, color='orange')
ax6.set_xlabel('Skupina postaj')
ax6.set_ylabel('Število primerov 2. VVS')
ax6.set_title('Število primerov 2. VVS')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(x_labels, rotation=20, ha='right')
ax6.legend(**legend_kwargs, loc='upper right')
ax6.grid(True, alpha=0.3)

# --- 3. VVS Natancnost (Precision) ---
ax7 = axes[2, 0]
ax7.bar(x_pos - width, df['TOK_vv3_precision'], width, label='TOK', alpha=0.8, color='skyblue')
ax7.bar(x_pos, df['arso_vv3_precision'], width, label='HPS', alpha=0.8, color='lightcoral')
ax7.bar(x_pos + width, df['linreg_vv3_precision'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax7.set_xlabel('Skupina postaj')
ax7.set_ylabel('Natančnost')
ax7.set_title('Natančnost napovedovanja 3. VVS')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(x_labels, rotation=20, ha='right')
ax7.set_ylim([0, 1])
ax7.legend(**legend_kwargs)
ax7.grid(True, alpha=0.3)

# --- 3. VVS Priklic (Recall) ---
ax8 = axes[2, 1]
ax8.bar(x_pos - width, df['TOK_vv3_recall'], width, label='TOK', alpha=0.8, color='skyblue')
ax8.bar(x_pos, df['arso_vv3_recall'], width, label='HPS', alpha=0.8, color='lightcoral')
ax8.bar(x_pos + width, df['linreg_vv3_recall'], width, label='Linearna regresija', alpha=0.8, color='orange')
ax8.set_xlabel('Skupina postaj')
ax8.set_ylabel('Priklic')
ax8.set_title('Priklic napovedovanja 3. VVS')
ax8.set_xticks(x_pos)
ax8.set_xticklabels(x_labels, rotation=20, ha='right')
ax8.set_ylim([0, 1])
ax8.legend(**legend_kwargs)
ax8.grid(True, alpha=0.3)

# --- 3. VVS GT Primeri ---
ax9 = axes[2, 2]
ax9.bar(x_pos, df['gt_vv3'], width, label='Primeri 3. VVS', alpha=0.8, color='red')
ax9.set_xlabel('Skupina postaj')
ax9.set_ylabel('Število primerov 3. VVS')
ax9.set_title('Število primerov 3. VVS')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(x_labels, rotation=20, ha='right')
ax9.legend(**legend_kwargs, loc='upper right')
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# MAP TIME TO DATETIME
# Loop over all rows in df_combined and convert 'index' column to datetime for each model_predictions_df
for i, row in df_combined.iterrows():
    modelpreddf = row['model_predictions_df']
    if modelpreddf is not None and 'index' in modelpreddf.columns:
        modelpreddf['index'] = pd.to_datetime(modelpreddf['index'], unit='ms')
        df_combined.at[i, 'model_predictions_df'] = modelpreddf


# read vv levels
vv_levels = pd.read_parquet('arso_vv.parquet')



for i, row in df_combined.iterrows():
    station_id = row['station_id']
    vv_levels_station = vv_levels[(vv_levels['station_id'].astype(int)) == station_id]

    
# For each run, check if actual Q in hours 4 onward surpasses any vv levels, and report the biggest vv passed (max vv3)
vv_report = []

# Group stations by first digit of station_id
station_groups = {}
for i, row in df_combined.iterrows():
    station_id = row['station_id']
    group_key = str(station_id)[0]  # First digit
    if group_key not in station_groups:
        station_groups[group_key] = []
    station_groups[group_key].append(i)

# Store results in a list for DataFrame creation
results_list = []

# Calculate VV statistics by group
for group_key, station_indices in station_groups.items():
    #print(f"\nGroup {group_key} (stations starting with {group_key}):")
    
    # Aggregate VV counts across all stations in this group
    group_vv_gt_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    group_vv_model_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    group_vv_arso_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    
    # Initialize confusion matrix statistics for each VV level
    # VV1 level (vv1+ vs no vv)
    model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn = 0, 0, 0, 0
    arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn = 0, 0, 0, 0
    
    # VV2 level (vv2+ vs vv1-)
    model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn = 0, 0, 0, 0
    arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn = 0, 0, 0, 0
    
    # VV3 level (vv3 vs vv2-)
    model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn = 0, 0, 0, 0
    arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn = 0, 0, 0, 0
    
    # Any VV level (any vv vs no vv)
    model_any_tp, model_any_tn, model_any_fp, model_any_fn = 0, 0, 0, 0
    arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn = 0, 0, 0, 0
    
    HOUR_CUTOFF = 3 # first 3 hours should not be already vv

    for i in station_indices:
        row = df_combined.iloc[i]
        actuals_df = row['actuals_df']
        station_id = row['station_id']
        vv_levels_station = vv_levels[vv_levels['station_id'].astype(int) == station_id]
        
        # Get vv1, vv2, vv3 for this station (if available)
        if len(vv_levels_station) == 0:
            continue
        vv1 = vv_levels_station['vv1'].values[0]
        vv2 = vv_levels_station['vv2'].values[0]
        vv3 = vv_levels_station['vv3'].values[0]

        rel = actuals_df[[str(i) for i in range(HOUR_CUTOFF, 18)]] # hourly predictions are 0...17
        rel_max = rel.max(axis=1)
        # For each row, assign the biggest vv level passed (vv3 > vv2 > vv1), or None if none passed
        def biggest_vv_passed(val):
            if val >= vv3:
                return 'vv3'
            elif val >= vv2:
                return 'vv2'
            elif val >= vv1:
                return 'vv1'
            else:
                return '-'
        biggest_vv_gt = rel_max.apply(biggest_vv_passed)
        rel_first6 = actuals_df[[str(i) for i in range(0, HOUR_CUTOFF)]] # hourly predictions are 0...17
        rel_first6_max = rel_first6.max(axis=1)
        biggest_vv_gt_first6 = rel_first6_max.apply(biggest_vv_passed)

        # Only set vv value if there was no vv in the first 6 hours
        biggest_vv_gt = biggest_vv_gt.where(biggest_vv_gt_first6 != '-', '-')

        # now look at model predictions and determine the vv
        model_predictions_df = row['model_predictions_df']
        rel_model = model_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
        rel_model_max = rel_model.max(axis=1)
        biggest_vv_model = rel_model_max.apply(biggest_vv_passed)

        # Only set vv value if there was no vv in the first 6 hours
        biggest_vv_model = biggest_vv_model.where(biggest_vv_gt_first6 != '-', '-')
        
        # now look at arso predictions and determine the vv
        arso_predictions_df = row['arso_predictions_df']
        rel_arso = arso_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
        rel_arso_max = rel_arso.max(axis=1)
        biggest_vv_arso = rel_arso_max.apply(biggest_vv_passed)
        biggest_vv_arso = biggest_vv_arso.where(biggest_vv_gt_first6 != '-', '-')

        # Add counts to group totals
        vv_gt_counts = biggest_vv_gt.value_counts()
        vv_model_counts = biggest_vv_model.value_counts()
        vv_arso_counts = biggest_vv_arso.value_counts()
        
        for vv_level in ['-', 'vv1', 'vv2', 'vv3']:
            group_vv_gt_counts[vv_level] += vv_gt_counts.get(vv_level, 0)
            group_vv_model_counts[vv_level] += vv_model_counts.get(vv_level, 0)
            group_vv_arso_counts[vv_level] += vv_arso_counts.get(vv_level, 0)
        
        # Calculate confusion matrix statistics for this station per VV level
        
        # VV1+ (vv1, vv2, vv3) vs no VV (-)
        gt_has_vv1_plus = (biggest_vv_gt.isin(['vv1']))
        model_has_vv1_plus = (biggest_vv_model.isin(['vv1']))
        arso_has_vv1_plus = (biggest_vv_arso.isin(['vv1']))
        
        model_vv1_tp += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == True)).sum()
        model_vv1_tn += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == False)).sum()
        model_vv1_fp += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == True)).sum()
        model_vv1_fn += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == False)).sum()
        
        arso_vv1_tp += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == True)).sum()
        arso_vv1_tn += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == False)).sum()
        arso_vv1_fp += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == True)).sum()
        arso_vv1_fn += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == False)).sum()
        
        # VV2+ (vv2, vv3) vs VV1- (-, vv1)
        gt_has_vv2_plus = (biggest_vv_gt.isin(['vv2']))
        model_has_vv2_plus = (biggest_vv_model.isin(['vv2']))
        arso_has_vv2_plus = (biggest_vv_arso.isin(['vv2']))
        
        model_vv2_tp += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == True)).sum()
        model_vv2_tn += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == False)).sum()
        model_vv2_fp += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == True)).sum()
        model_vv2_fn += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == False)).sum()
        
        arso_vv2_tp += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == True)).sum()
        arso_vv2_tn += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == False)).sum()
        arso_vv2_fp += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == True)).sum()
        arso_vv2_fn += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == False)).sum()
        
        # VV3 vs VV2- (-, vv1, vv2)
        gt_has_vv3 = (biggest_vv_gt == 'vv3')
        model_has_vv3 = (biggest_vv_model == 'vv3')
        arso_has_vv3 = (biggest_vv_arso == 'vv3')
        
        model_vv3_tp += ((gt_has_vv3 == True) & (model_has_vv3 == True)).sum()
        model_vv3_tn += ((gt_has_vv3 == False) & (model_has_vv3 == False)).sum()
        model_vv3_fp += ((gt_has_vv3 == False) & (model_has_vv3 == True)).sum()
        model_vv3_fn += ((gt_has_vv3 == True) & (model_has_vv3 == False)).sum()
        
        arso_vv3_tp += ((gt_has_vv3 == True) & (arso_has_vv3 == True)).sum()
        arso_vv3_tn += ((gt_has_vv3 == False) & (arso_has_vv3 == False)).sum()
        arso_vv3_fp += ((gt_has_vv3 == False) & (arso_has_vv3 == True)).sum()
        arso_vv3_fn += ((gt_has_vv3 == True) & (arso_has_vv3 == False)).sum()
        
        # Any VV (vv1, vv2, vv3) vs no VV (-)
        gt_has_any_vv = (biggest_vv_gt != '-')
        model_has_any_vv = (biggest_vv_model != '-')
        arso_has_any_vv = (biggest_vv_arso != '-')
        
        model_any_tp += ((gt_has_any_vv == True) & (model_has_any_vv == True)).sum()
        model_any_tn += ((gt_has_any_vv == False) & (model_has_any_vv == False)).sum()
        model_any_fp += ((gt_has_any_vv == False) & (model_has_any_vv == True)).sum()
        model_any_fn += ((gt_has_any_vv == True) & (model_has_any_vv == False)).sum()
        
        arso_any_tp += ((gt_has_any_vv == True) & (arso_has_any_vv == True)).sum()
        arso_any_tn += ((gt_has_any_vv == False) & (arso_has_any_vv == False)).sum()
        arso_any_fp += ((gt_has_any_vv == False) & (arso_has_any_vv == True)).sum()
        arso_any_fn += ((gt_has_any_vv == True) & (arso_has_any_vv == False)).sum()
    
    # Calculate metrics for each VV level
    def calculate_metrics(tp, tn, fp, fn):
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        return precision, recall
    
    # VV1+ metrics
    model_vv1_precision, model_vv1_recall = calculate_metrics(model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn)
    arso_vv1_precision, arso_vv1_recall = calculate_metrics(arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn)
    
    # VV2+ metrics
    model_vv2_precision, model_vv2_recall = calculate_metrics(model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn)
    arso_vv2_precision, arso_vv2_recall = calculate_metrics(arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn)
    
    # VV3 metrics
    model_vv3_precision, model_vv3_recall = calculate_metrics(model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn)
    arso_vv3_precision, arso_vv3_recall = calculate_metrics(arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn)
    
    # Any VV metrics
    model_any_precision, model_any_recall = calculate_metrics(model_any_tp, model_any_tn, model_any_fp, model_any_fn)
    arso_any_precision, arso_any_recall = calculate_metrics(arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn)
    
    # Store results for this group
    results_list.append({
        'group': group_key,
        'gt_no_vv': group_vv_gt_counts['-'],
        'gt_vv1': group_vv_gt_counts['vv1'],
        'gt_vv2': group_vv_gt_counts['vv2'],
        'gt_vv3': group_vv_gt_counts['vv3'],
        'model_no_vv': group_vv_model_counts['-'],
        'model_vv1': group_vv_model_counts['vv1'],
        'model_vv2': group_vv_model_counts['vv2'],
        'model_vv3': group_vv_model_counts['vv3'],
        'arso_no_vv': group_vv_arso_counts['-'],
        'arso_vv1': group_vv_arso_counts['vv1'],
        'arso_vv2': group_vv_arso_counts['vv2'],
        'arso_vv3': group_vv_arso_counts['vv3'],
        # VV1+ metrics
        'model_vv1_tp': model_vv1_tp, 'model_vv1_tn': model_vv1_tn, 'model_vv1_fp': model_vv1_fp, 'model_vv1_fn': model_vv1_fn,
        'model_vv1_precision': model_vv1_precision, 'model_vv1_recall': model_vv1_recall,
        'arso_vv1_tp': arso_vv1_tp, 'arso_vv1_tn': arso_vv1_tn, 'arso_vv1_fp': arso_vv1_fp, 'arso_vv1_fn': arso_vv1_fn,
        'arso_vv1_precision': arso_vv1_precision, 'arso_vv1_recall': arso_vv1_recall,
        # VV2+ metrics
        'model_vv2_tp': model_vv2_tp, 'model_vv2_tn': model_vv2_tn, 'model_vv2_fp': model_vv2_fp, 'model_vv2_fn': model_vv2_fn,
        'model_vv2_precision': model_vv2_precision, 'model_vv2_recall': model_vv2_recall,
        'arso_vv2_tp': arso_vv2_tp, 'arso_vv2_tn': arso_vv2_tn, 'arso_vv2_fp': arso_vv2_fp, 'arso_vv2_fn': arso_vv2_fn,
        'arso_vv2_precision': arso_vv2_precision, 'arso_vv2_recall': arso_vv2_recall,
        # VV3 metrics
        'model_vv3_tp': model_vv3_tp, 'model_vv3_tn': model_vv3_tn, 'model_vv3_fp': model_vv3_fp, 'model_vv3_fn': model_vv3_fn,
        'model_vv3_precision': model_vv3_precision, 'model_vv3_recall': model_vv3_recall,
        'arso_vv3_tp': arso_vv3_tp, 'arso_vv3_tn': arso_vv3_tn, 'arso_vv3_fp': arso_vv3_fp, 'arso_vv3_fn': arso_vv3_fn,
        'arso_vv3_precision': arso_vv3_precision, 'arso_vv3_recall': arso_vv3_recall,
        # Any VV metrics
        'model_any_tp': model_any_tp, 'model_any_tn': model_any_tn, 'model_any_fp': model_any_fp, 'model_any_fn': model_any_fn,
        'model_any_precision': model_any_precision, 'model_any_recall': model_any_recall,
        'arso_any_tp': arso_any_tp, 'arso_any_tn': arso_any_tn, 'arso_any_fp': arso_any_fp, 'arso_any_fn': arso_any_fn,
        'arso_any_precision': arso_any_precision, 'arso_any_recall': arso_any_recall
    })

# Create DataFrame from results
vv_results_df = pd.DataFrame(results_list)
vv_results_df_TOK = vv_results_df.copy()

# Display the results table
print("VV Warning Performance by Station Group and VV Level:")
print("="*80)
display(vv_results_df)

# Create visualizations
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('VV Warning Performance Analysis by Station Group and VV Level', fontsize=16, fontweight='bold')

x_pos = np.arange(len(vv_results_df))
width = 0.35

# VV1+ Analysis
ax1 = axes[0, 0]
ax1.bar(x_pos - width/2, vv_results_df['model_vv1_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax1.bar(x_pos + width/2, vv_results_df['arso_vv1_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax1.set_xlabel('Station Group')
ax1.set_ylabel('Precision')
ax1.set_title('VV1+ Precision Comparison')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(vv_results_df['group'])
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.bar(x_pos - width/2, vv_results_df['model_vv1_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax2.bar(x_pos + width/2, vv_results_df['arso_vv1_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax2.set_xlabel('Station Group')
ax2.set_ylabel('Recall')
ax2.set_title('VV1+ Recall Comparison')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(vv_results_df['group'])
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
ax3.bar(x_pos, vv_results_df['gt_vv1'], width, 
        label='Ground Truth VV1 Cases', alpha=0.8, color='green')
ax3.set_xlabel('Station Group')
ax3.set_ylabel('Number of VV1 Cases')
ax3.set_title('VV1 Cases per Station Group')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(vv_results_df['group'])
ax3.legend()
ax3.grid(True, alpha=0.3)

# VV2+ Analysis
ax4 = axes[1, 0]
ax4.bar(x_pos - width/2, vv_results_df['model_vv2_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax4.bar(x_pos + width/2, vv_results_df['arso_vv2_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax4.set_xlabel('Station Group')
ax4.set_ylabel('Precision')
ax4.set_title('VV2+ Precision Comparison')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(vv_results_df['group'])
ax4.legend()
ax4.grid(True, alpha=0.3)

ax5 = axes[1, 1]
ax5.bar(x_pos - width/2, vv_results_df['model_vv2_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax5.bar(x_pos + width/2, vv_results_df['arso_vv2_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax5.set_xlabel('Station Group')
ax5.set_ylabel('Recall')
ax5.set_title('VV2+ Recall Comparison')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(vv_results_df['group'])
ax5.legend()
ax5.grid(True, alpha=0.3)

ax6 = axes[1, 2]
ax6.bar(x_pos, vv_results_df['gt_vv2'], width, 
        label='Ground Truth VV2 Cases', alpha=0.8, color='orange')
ax6.set_xlabel('Station Group')
ax6.set_ylabel('Number of VV2 Cases')
ax6.set_title('VV2 Cases per Station Group')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(vv_results_df['group'])
ax6.legend()
ax6.grid(True, alpha=0.3)

# VV3 Analysis
ax7 = axes[2, 0]
ax7.bar(x_pos - width/2, vv_results_df['model_vv3_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax7.bar(x_pos + width/2, vv_results_df['arso_vv3_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax7.set_xlabel('Station Group')
ax7.set_ylabel('Precision')
ax7.set_title('VV3 Precision Comparison')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(vv_results_df['group'])
ax7.legend()
ax7.grid(True, alpha=0.3)

ax8 = axes[2, 1]
ax8.bar(x_pos - width/2, vv_results_df['model_vv3_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax8.bar(x_pos + width/2, vv_results_df['arso_vv3_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax8.set_xlabel('Station Group')
ax8.set_ylabel('Recall')
ax8.set_title('VV3 Recall Comparison')
ax8.set_xticks(x_pos)
ax8.set_xticklabels(vv_results_df['group'])
ax8.legend()
ax8.grid(True, alpha=0.3)

ax9 = axes[2, 2]
ax9.bar(x_pos, vv_results_df['gt_vv3'], width, 
        label='Ground Truth VV3 Cases', alpha=0.8, color='red')
ax9.set_xlabel('Station Group')
ax9.set_ylabel('Number of VV3 Cases')
ax9.set_title('VV3 Cases per Station Group')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(vv_results_df['group'])
ax9.legend()
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics per VV level
print("\nSummary Statistics by VV Level:")
print("-" * 60)
print("VV1+ (Any VV1 or higher):")
print(f"  Model Precision: {vv_results_df['model_vv1_precision'].mean():.3f} ± {vv_results_df['model_vv1_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv1_precision'].mean():.3f} ± {vv_results_df['arso_vv1_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv1_recall'].mean():.3f} ± {vv_results_df['model_vv1_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv1_recall'].mean():.3f} ± {vv_results_df['arso_vv1_recall'].std():.3f}")
print(f"  Total VV1 Cases: {vv_results_df['gt_vv1'].sum()}")

print("\nVV2+ (VV2 or VV3):")
print(f"  Model Precision: {vv_results_df['model_vv2_precision'].mean():.3f} ± {vv_results_df['model_vv2_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv2_precision'].mean():.3f} ± {vv_results_df['arso_vv2_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv2_recall'].mean():.3f} ± {vv_results_df['model_vv2_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv2_recall'].mean():.3f} ± {vv_results_df['arso_vv2_recall'].std():.3f}")
print(f"  Total VV2 Cases: {vv_results_df['gt_vv2'].sum()}")

print("\nVV3 (Highest level):")
print(f"  Model Precision: {vv_results_df['model_vv3_precision'].mean():.3f} ± {vv_results_df['model_vv3_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv3_precision'].mean():.3f} ± {vv_results_df['arso_vv3_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv3_recall'].mean():.3f} ± {vv_results_df['model_vv3_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv3_recall'].mean():.3f} ± {vv_results_df['arso_vv3_recall'].std():.3f}")
print(f"  Total VV3 Cases: {vv_results_df['gt_vv3'].sum()}")

print("\nAny VV (Overall):")
print(f"  Model Precision: {vv_results_df['model_any_precision'].mean():.3f} ± {vv_results_df['model_any_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_any_precision'].mean():.3f} ± {vv_results_df['arso_any_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_any_recall'].mean():.3f} ± {vv_results_df['model_any_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_any_recall'].mean():.3f} ± {vv_results_df['arso_any_recall'].std():.3f}")
print(f"  Total Any VV Cases: {vv_results_df['gt_vv1'].sum() + vv_results_df['gt_vv2'].sum() + vv_results_df['gt_vv3'].sum()}")


In [ ]:
# MAP TIME TO DATETIME
# Loop over all rows in df_combined and convert 'index' column to datetime for each model_predictions_df
for i, row in df_combined.iterrows():
    modelpreddf = row['model_predictions_df_linreg']
    if modelpreddf is not None and 'index' in modelpreddf.columns:
        modelpreddf['index'] = pd.to_datetime(modelpreddf['index'], unit='ms')
        df_combined.at[i, 'model_predictions_df_linreg'] = modelpreddf


# read vv levels
vv_levels = pd.read_parquet('arso_vv.parquet')



for i, row in df_combined.iterrows():
    station_id = row['station_id']
    vv_levels_station = vv_levels[(vv_levels['station_id'].astype(int)) == station_id]

    
# For each run, check if actual Q in hours 4 onward surpasses any vv levels, and report the biggest vv passed (max vv3)
vv_report = []

# Group stations by first digit of station_id
station_groups = {}
for i, row in df_combined.iterrows():
    station_id = row['station_id']
    group_key = str(station_id)[0]  # First digit
    if group_key not in station_groups:
        station_groups[group_key] = []
    station_groups[group_key].append(i)

# Store results in a list for DataFrame creation
results_list = []

# Calculate VV statistics by group
for group_key, station_indices in station_groups.items():
    #print(f"\nGroup {group_key} (stations starting with {group_key}):")
    
    # Aggregate VV counts across all stations in this group
    group_vv_gt_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    group_vv_model_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    group_vv_arso_counts = {'-': 0, 'vv1': 0, 'vv2': 0, 'vv3': 0}
    
    # Initialize confusion matrix statistics for each VV level
    # VV1 level (vv1+ vs no vv)
    model_vv1_tp, model_vv1_tn, model_vv1_fp, model_vv1_fn = 0, 0, 0, 0
    arso_vv1_tp, arso_vv1_tn, arso_vv1_fp, arso_vv1_fn = 0, 0, 0, 0
    
    # VV2 level (vv2+ vs vv1-)
    model_vv2_tp, model_vv2_tn, model_vv2_fp, model_vv2_fn = 0, 0, 0, 0
    arso_vv2_tp, arso_vv2_tn, arso_vv2_fp, arso_vv2_fn = 0, 0, 0, 0
    
    # VV3 level (vv3 vs vv2-)
    model_vv3_tp, model_vv3_tn, model_vv3_fp, model_vv3_fn = 0, 0, 0, 0
    arso_vv3_tp, arso_vv3_tn, arso_vv3_fp, arso_vv3_fn = 0, 0, 0, 0
    
    # Any VV level (any vv vs no vv)
    model_any_tp, model_any_tn, model_any_fp, model_any_fn = 0, 0, 0, 0
    arso_any_tp, arso_any_tn, arso_any_fp, arso_any_fn = 0, 0, 0, 0
    
    HOUR_CUTOFF = 3 # first 3 hours should not be already vv

    for i in station_indices:
        row = df_combined.iloc[i]
        actuals_df = row['actuals_df_linreg']
        station_id = row['station_id']
        vv_levels_station = vv_levels[vv_levels['station_id'].astype(int) == station_id]
        
        # Get vv1, vv2, vv3 for this station (if available)
        if len(vv_levels_station) == 0:
            continue
        vv1 = vv_levels_station['vv1'].values[0]
        vv2 = vv_levels_station['vv2'].values[0]
        vv3 = vv_levels_station['vv3'].values[0]

        rel = actuals_df[[str(i) for i in range(HOUR_CUTOFF, 18)]] # hourly predictions are 0...17
        rel_max = rel.max(axis=1)
        # For each row, assign the biggest vv level passed (vv3 > vv2 > vv1), or None if none passed
        def biggest_vv_passed(val):
            if val >= vv3:
                return 'vv3'
            elif val >= vv2:
                return 'vv2'
            elif val >= vv1:
                return 'vv1'
            else:
                return '-'
        biggest_vv_gt = rel_max.apply(biggest_vv_passed)
        rel_first6 = actuals_df[[str(i) for i in range(0, HOUR_CUTOFF)]] # hourly predictions are 0...17
        rel_first6_max = rel_first6.max(axis=1)
        biggest_vv_gt_first6 = rel_first6_max.apply(biggest_vv_passed)

        # Only set vv value if there was no vv in the first 6 hours
        biggest_vv_gt = biggest_vv_gt.where(biggest_vv_gt_first6 == '-', '-')

        # now look at model predictions and determine the vv
        model_predictions_df = row['model_predictions_df_linreg']
        rel_model = model_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
        rel_model_max = rel_model.max(axis=1)
        biggest_vv_model = rel_model_max.apply(biggest_vv_passed)

        # Only set vv value if there was no vv in the first 6 hours
        biggest_vv_model = biggest_vv_model.where(biggest_vv_gt_first6 == '-', '-')
        
        # now look at arso predictions and determine the vv
        arso_predictions_df = row['arso_predictions_df_linreg']
        rel_arso = arso_predictions_df[[str(i) for i in range(HOUR_CUTOFF, 18)]]
        rel_arso_max = rel_arso.max(axis=1)
        biggest_vv_arso = rel_arso_max.apply(biggest_vv_passed)
        biggest_vv_arso = biggest_vv_arso.where(biggest_vv_gt_first6 == '-', '-')

        # Add counts to group totals
        vv_gt_counts = biggest_vv_gt.value_counts()
        vv_model_counts = biggest_vv_model.value_counts()
        vv_arso_counts = biggest_vv_arso.value_counts()
        
        for vv_level in ['-', 'vv1', 'vv2', 'vv3']:
            group_vv_gt_counts[vv_level] += vv_gt_counts.get(vv_level, 0)
            group_vv_model_counts[vv_level] += vv_model_counts.get(vv_level, 0)
            group_vv_arso_counts[vv_level] += vv_arso_counts.get(vv_level, 0)
        
        # Calculate confusion matrix statistics for this station per VV level
        
        # VV1+ (vv1, vv2, vv3) vs no VV (-)
        gt_has_vv1_plus = (biggest_vv_gt.isin(['vv1']))
        model_has_vv1_plus = (biggest_vv_model.isin(['vv1']))
        arso_has_vv1_plus = (biggest_vv_arso.isin(['vv1']))
        
        model_vv1_tp += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == True)).sum()
        model_vv1_tn += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == False)).sum()
        model_vv1_fp += ((gt_has_vv1_plus == False) & (model_has_vv1_plus == True)).sum()
        model_vv1_fn += ((gt_has_vv1_plus == True) & (model_has_vv1_plus == False)).sum()
        
        arso_vv1_tp += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == True)).sum()
        arso_vv1_tn += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == False)).sum()
        arso_vv1_fp += ((gt_has_vv1_plus == False) & (arso_has_vv1_plus == True)).sum()
        arso_vv1_fn += ((gt_has_vv1_plus == True) & (arso_has_vv1_plus == False)).sum()
        
        # VV2+ (vv2, vv3) vs VV1- (-, vv1)
        gt_has_vv2_plus = (biggest_vv_gt.isin(['vv2']))
        model_has_vv2_plus = (biggest_vv_model.isin(['vv2']))
        arso_has_vv2_plus = (biggest_vv_arso.isin(['vv2']))
        
        model_vv2_tp += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == True)).sum()
        model_vv2_tn += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == False)).sum()
        model_vv2_fp += ((gt_has_vv2_plus == False) & (model_has_vv2_plus == True)).sum()
        model_vv2_fn += ((gt_has_vv2_plus == True) & (model_has_vv2_plus == False)).sum()
        
        arso_vv2_tp += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == True)).sum()
        arso_vv2_tn += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == False)).sum()
        arso_vv2_fp += ((gt_has_vv2_plus == False) & (arso_has_vv2_plus == True)).sum()
        arso_vv2_fn += ((gt_has_vv2_plus == True) & (arso_has_vv2_plus == False)).sum()
        
        # VV3 (vv3) vs VV2- (-, vv1, vv2)
        gt_has_vv3 = (biggest_vv_gt == 'vv3')
        model_has_vv3 = (biggest_vv_model == 'vv3')
        arso_has_vv3 = (biggest_vv_arso == 'vv3')
        
        model_vv3_tp += ((gt_has_vv3 == True) & (model_has_vv3 == True)).sum()
        model_vv3_tn += ((gt_has_vv3 == False) & (model_has_vv3 == False)).sum()
        model_vv3_fp += ((gt_has_vv3 == False) & (model_has_vv3 == True)).sum()
        model_vv3_fn += ((gt_has_vv3 == True) & (model_has_vv3 == False)).sum()
        
        arso_vv3_tp += ((gt_has_vv3 == True) & (arso_has_vv3 == True)).sum()
        arso_vv3_tn += ((gt_has_vv3 == False) & (arso_has_vv3 == False)).sum()
        arso_vv3_fp += ((gt_has_vv3 == False) & (arso_has_vv3 == True)).sum()
        arso_vv3_fn += ((gt_has_vv3 == True) & (arso_has_vv3 == False)).sum()
        
        # Any VV (vv1, vv2, vv3) vs no VV (-)
        gt_has_any_vv = (biggest_vv_gt != '-')
        model_has_any_vv = (biggest_vv_model != '-')
        arso_has_any_vv = (biggest_vv_arso != '-')
        
        model_any_tp += ((gt_has_any_vv == True) & (model_has_any_vv == True)).sum()
        model_any_tn += ((gt_has_any_vv == False) & (model_has_any_vv == False)).sum()
        model_any_fp += ((gt_has_any_vv == False) & (model_has_any_vv == True)).sum()
        model_any_fn += ((gt_has_any_vv == True) & (model_has_any_vv == False)).sum()
        
        arso_any_tp += ((gt_has_any_vv == True) & (arso_has_any_vv == True)).sum()
        arso_any_tn += ((gt_has_any_vv == False) & (arso_has_any_vv == False)).sum()
        arso_any_fp += ((gt_has_any_vv == False) & (arso_has_any_vv == True)).sum()
        arso_any_fn += ((gt_has_any_vv == True) & (arso_has_any_vv == False)).sum()
    
    # Calculate metrics for each VV level
    def safe_divide(a, b):
        return a / b if b > 0 else 0
    
    # VV1+ metrics
    model_vv1_precision = safe_divide(model_vv1_tp, model_vv1_tp + model_vv1_fp)
    model_vv1_recall = safe_divide(model_vv1_tp, model_vv1_tp + model_vv1_fn)
    model_vv1_accuracy = safe_divide(model_vv1_tp + model_vv1_tn, model_vv1_tp + model_vv1_tn + model_vv1_fp + model_vv1_fn)
    
    arso_vv1_precision = safe_divide(arso_vv1_tp, arso_vv1_tp + arso_vv1_fp)
    arso_vv1_recall = safe_divide(arso_vv1_tp, arso_vv1_tp + arso_vv1_fn)
    arso_vv1_accuracy = safe_divide(arso_vv1_tp + arso_vv1_tn, arso_vv1_tp + arso_vv1_tn + arso_vv1_fp + arso_vv1_fn)
    
    # VV2+ metrics
    model_vv2_precision = safe_divide(model_vv2_tp, model_vv2_tp + model_vv2_fp)
    model_vv2_recall = safe_divide(model_vv2_tp, model_vv2_tp + model_vv2_fn)
    model_vv2_accuracy = safe_divide(model_vv2_tp + model_vv2_tn, model_vv2_tp + model_vv2_tn + model_vv2_fp + model_vv2_fn)
    
    arso_vv2_precision = safe_divide(arso_vv2_tp, arso_vv2_tp + arso_vv2_fp)
    arso_vv2_recall = safe_divide(arso_vv2_tp, arso_vv2_tp + arso_vv2_fn)
    arso_vv2_accuracy = safe_divide(arso_vv2_tp + arso_vv2_tn, arso_vv2_tp + arso_vv2_tn + arso_vv2_fp + arso_vv2_fn)
    
    # VV3 metrics
    model_vv3_precision = safe_divide(model_vv3_tp, model_vv3_tp + model_vv3_fp)
    model_vv3_recall = safe_divide(model_vv3_tp, model_vv3_tp + model_vv3_fn)
    model_vv3_accuracy = safe_divide(model_vv3_tp + model_vv3_tn, model_vv3_tp + model_vv3_tn + model_vv3_fp + model_vv3_fn)
    
    arso_vv3_precision = safe_divide(arso_vv3_tp, arso_vv3_tp + arso_vv3_fp)
    arso_vv3_recall = safe_divide(arso_vv3_tp, arso_vv3_tp + arso_vv3_fn)
    arso_vv3_accuracy = safe_divide(arso_vv3_tp + arso_vv3_tn, arso_vv3_tp + arso_vv3_tn + arso_vv3_fp + arso_vv3_fn)
    
    # Any VV metrics
    model_any_precision = safe_divide(model_any_tp, model_any_tp + model_any_fp)
    model_any_recall = safe_divide(model_any_tp, model_any_tp + model_any_fn)
    model_any_accuracy = safe_divide(model_any_tp + model_any_tn, model_any_tp + model_any_tn + model_any_fp + model_any_fn)
    
    arso_any_precision = safe_divide(arso_any_tp, arso_any_tp + arso_any_fp)
    arso_any_recall = safe_divide(arso_any_tp, arso_any_tp + arso_any_fn)
    arso_any_accuracy = safe_divide(arso_any_tp + arso_any_tn, arso_any_tp + arso_any_tn + arso_any_fp + arso_any_fn)
    
    # Store results
    results_list.append({
        'group': group_key,
        'model_vv1_precision': model_vv1_precision,
        'model_vv1_recall': model_vv1_recall,
        'model_vv1_accuracy': model_vv1_accuracy,
        'arso_vv1_precision': arso_vv1_precision,
        'arso_vv1_recall': arso_vv1_recall,
        'arso_vv1_accuracy': arso_vv1_accuracy,
        'model_vv2_precision': model_vv2_precision,
        'model_vv2_recall': model_vv2_recall,
        'model_vv2_accuracy': model_vv2_accuracy,
        'arso_vv2_precision': arso_vv2_precision,
        'arso_vv2_recall': arso_vv2_recall,
        'arso_vv2_accuracy': arso_vv2_accuracy,
        'model_vv3_precision': model_vv3_precision,
        'model_vv3_recall': model_vv3_recall,
        'model_vv3_accuracy': model_vv3_accuracy,
        'arso_vv3_precision': arso_vv3_precision,
        'arso_vv3_recall': arso_vv3_recall,
        'arso_vv3_accuracy': arso_vv3_accuracy,
        'model_any_precision': model_any_precision,
        'model_any_recall': model_any_recall,
        'model_any_accuracy': model_any_accuracy,
        'arso_any_precision': arso_any_precision,
        'arso_any_recall': arso_any_recall,
        'arso_any_accuracy': arso_any_accuracy
    })

# Create DataFrame from results
vv_results_df = pd.DataFrame(results_list)
print(vv_results_df)

# Create visualization
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
x_pos = np.arange(len(vv_results_df))
width = 0.35

# VV1+ Analysis
ax1 = axes[0, 0]
ax1.bar(x_pos - width/2, vv_results_df['model_vv1_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax1.bar(x_pos + width/2, vv_results_df['arso_vv1_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax1.set_xlabel('Station Group')
ax1.set_ylabel('Precision')
ax1.set_title('VV1+ Precision Comparison')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(vv_results_df['group'])
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.bar(x_pos - width/2, vv_results_df['model_vv1_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax2.bar(x_pos + width/2, vv_results_df['arso_vv1_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax2.set_xlabel('Station Group')
ax2.set_ylabel('Recall')
ax2.set_title('VV1+ Recall Comparison')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(vv_results_df['group'])
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
ax3.bar(x_pos - width/2, vv_results_df['model_vv1_accuracy'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax3.bar(x_pos + width/2, vv_results_df['arso_vv1_accuracy'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax3.set_xlabel('Station Group')
ax3.set_ylabel('Accuracy')
ax3.set_title('VV1+ Accuracy Comparison')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(vv_results_df['group'])
ax3.legend()
ax3.grid(True, alpha=0.3)

# VV2+ Analysis
ax4 = axes[1, 0]
ax4.bar(x_pos - width/2, vv_results_df['model_vv2_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax4.bar(x_pos + width/2, vv_results_df['arso_vv2_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax4.set_xlabel('Station Group')
ax4.set_ylabel('Precision')
ax4.set_title('VV2+ Precision Comparison')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(vv_results_df['group'])
ax4.legend()
ax4.grid(True, alpha=0.3)

ax5 = axes[1, 1]
ax5.bar(x_pos - width/2, vv_results_df['model_vv2_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax5.bar(x_pos + width/2, vv_results_df['arso_vv2_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax5.set_xlabel('Station Group')
ax5.set_ylabel('Recall')
ax5.set_title('VV2+ Recall Comparison')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(vv_results_df['group'])
ax5.legend()
ax5.grid(True, alpha=0.3)

ax6 = axes[1, 2]
ax6.bar(x_pos - width/2, vv_results_df['model_vv2_accuracy'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax6.bar(x_pos + width/2, vv_results_df['arso_vv2_accuracy'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax6.set_xlabel('Station Group')
ax6.set_ylabel('Accuracy')
ax6.set_title('VV2+ Accuracy Comparison')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(vv_results_df['group'])
ax6.legend()
ax6.grid(True, alpha=0.3)

# VV3 Analysis
ax7 = axes[2, 0]
ax7.bar(x_pos - width/2, vv_results_df['model_vv3_precision'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax7.bar(x_pos + width/2, vv_results_df['arso_vv3_precision'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax7.set_xlabel('Station Group')
ax7.set_ylabel('Precision')
ax7.set_title('VV3 Precision Comparison')
ax7.set_xticks(x_pos)
ax7.set_xticklabels(vv_results_df['group'])
ax7.legend()
ax7.grid(True, alpha=0.3)

ax8 = axes[2, 1]
ax8.bar(x_pos - width/2, vv_results_df['model_vv3_recall'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax8.bar(x_pos + width/2, vv_results_df['arso_vv3_recall'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax8.set_xlabel('Station Group')
ax8.set_ylabel('Recall')
ax8.set_title('VV3 Recall Comparison')
ax8.set_xticks(x_pos)
ax8.set_xticklabels(vv_results_df['group'])
ax8.legend()
ax8.grid(True, alpha=0.3)

ax9 = axes[2, 2]
ax9.bar(x_pos - width/2, vv_results_df['model_vv3_accuracy'], width, 
        label='Model', alpha=0.8, color='skyblue')
ax9.bar(x_pos + width/2, vv_results_df['arso_vv3_accuracy'], width, 
        label='ARSO', alpha=0.8, color='lightcoral')
ax9.set_xlabel('Station Group')
ax9.set_ylabel('Accuracy')
ax9.set_title('VV3 Accuracy Comparison')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(vv_results_df['group'])
ax9.legend()
ax9.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics per VV level
print("\nSummary Statistics by VV Level:")
print("-" * 60)
print("VV1+ (Any VV1 or higher):")
print(f"  Model Precision: {vv_results_df['model_vv1_precision'].mean():.3f} ± {vv_results_df['model_vv1_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv1_precision'].mean():.3f} ± {vv_results_df['arso_vv1_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv1_recall'].mean():.3f} ± {vv_results_df['model_vv1_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv1_recall'].mean():.3f} ± {vv_results_df['arso_vv1_recall'].std():.3f}")

print("\nVV2+ (VV2 or VV3):")
print(f"  Model Precision: {vv_results_df['model_vv2_precision'].mean():.3f} ± {vv_results_df['model_vv2_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv2_precision'].mean():.3f} ± {vv_results_df['arso_vv2_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv2_recall'].mean():.3f} ± {vv_results_df['model_vv2_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv2_recall'].mean():.3f} ± {vv_results_df['arso_vv2_recall'].std():.3f}")

print("\nVV3 (Highest level):")
print(f"  Model Precision: {vv_results_df['model_vv3_precision'].mean():.3f} ± {vv_results_df['model_vv3_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_vv3_precision'].mean():.3f} ± {vv_results_df['arso_vv3_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_vv3_recall'].mean():.3f} ± {vv_results_df['model_vv3_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_vv3_recall'].mean():.3f} ± {vv_results_df['arso_vv3_recall'].std():.3f}")

print("\nAny VV (Overall):")
print(f"  Model Precision: {vv_results_df['model_any_precision'].mean():.3f} ± {vv_results_df['model_any_precision'].std():.3f}")
print(f"  ARSO Precision: {vv_results_df['arso_any_precision'].mean():.3f} ± {vv_results_df['arso_any_precision'].std():.3f}")
print(f"  Model Recall: {vv_results_df['model_any_recall'].mean():.3f} ± {vv_results_df['model_any_recall'].std():.3f}")
print(f"  ARSO Recall: {vv_results_df['arso_any_recall'].mean():.3f} ± {vv_results_df['arso_any_recall'].std():.3f}")


In [ ]:
arso_pred_df = row['arso_predictions_df']
arso_pred_df = arso_pred_df.set_index('index')
arso_pred_df.index = pd.to_datetime(arso_pred_df.index)
print(f"Shapes: {actuals_df.shape}, {arso_pred_df.shape}, {model_pred_df.shape}")
print(f"Date ranges:")
print(f"Actuals: {actuals_df.index.min()} to {actuals_df.index.max()}")
print(f"ARSO: {arso_pred_df.index.min()} to {arso_pred_df.index.max()}")
print(f"Model: {model_pred_df.index.min()} to {model_pred_df.index.max()}")

In [ ]:
def create_interactive_plots(arso_pred_df, model_pred_df, actuals_df, name):
    import ipywidgets as widgets
    from IPython.display import display
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    from datetime import datetime
    """
    Create interactive plots for comparing predictions across dates and hours.
    
    Args:
        arso_pred_df: DataFrame with ARSO predictions
        model_pred_df: DataFrame with model predictions
        actuals_df: DataFrame with actual values
    """
    def plot_predictions_for_date(date_idx, arso_pred_df, model_pred_df, actuals_df, name):
        """Plot predictions for a specific date from all three dataframes."""
        dates = actuals_df.index.unique()
        if date_idx >= len(dates):
            print(f"Date index {date_idx} out of range. Max index: {len(dates)-1}")
            return
        
        selected_date = dates[date_idx]
        
        arso_data = arso_pred_df.loc[selected_date] if selected_date in arso_pred_df.index else None
        model_data = model_pred_df.loc[selected_date] if selected_date in model_pred_df.index else None
        actual_data = actuals_df.loc[selected_date] if selected_date in actuals_df.index else None
        
        fig, ax = plt.subplots(figsize=(12, 6))
        hours = range(1,19)
        
        if actual_data is not None:
            ax.plot(hours, actual_data.values, 'k-', linewidth=2, label='Merjene vrednosti')
        if model_data is not None:
            ax.plot(hours, model_data.values, 'b-', linewidth=2, label='TOK napoved')
        if arso_data is not None:
            ax.plot(hours, arso_data.values, 'r-', linewidth=2, label='HPS napoved')
        
        ax.set_xticks(hours)
        ax.set_xlabel('Napovedna ura')
        ax.set_ylabel('Pretok (m³/s)')
        ax.set_title(f'Napovedi za {selected_date.strftime("%Y-%m-%d")} - postaja {name}')
        ax.grid(True)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        return fig

    def save_predictions_for_date(date_idx, arso_pred_df, model_pred_df, actuals_df, filename=None):
        """Save plot figure for a specific date."""
        dates = actuals_df.index.unique()
        if date_idx >= len(dates):
            print(f"Date index {date_idx} out of range. Max index: {len(dates)-1}")
            return
        
        selected_date = dates[date_idx]
        
        # Get data for the selected date
        arso_data = arso_pred_df.loc[selected_date] if selected_date in arso_pred_df.index else None
        model_data = model_pred_df.loc[selected_date] if selected_date in model_pred_df.index else None
        actual_data = actuals_df.loc[selected_date] if selected_date in actuals_df.index else None
        
        # Generate filename if not provided
        if filename is None:
            filename = f"predictions_{selected_date.strftime('%Y-%m-%d')}.png"
        elif not filename.endswith('.png'):
            filename = filename + '.png'
        import os
        filepath = os.path.join(FIGURES_DIR, filename)

        # Create and save plot
        fig, ax = plt.subplots(figsize=(12, 6))
        hours = range(1, 19)
        
        if actual_data is not None:
            ax.plot(hours, actual_data.values, 'k-', linewidth=2, label='Merjene vrednosti')
        if model_data is not None:
            ax.plot(hours, model_data.values, 'b-', linewidth=2, label='TOK napoved')
        if arso_data is not None:
            ax.plot(hours, arso_data.values, 'r-', linewidth=2, label='HPS napoved')
        
        ax.set_xticks(hours)
        ax.set_xlabel('Napovedna ura')
        ax.set_ylabel('Pretok (m³/s)')
        ax.set_title(f'Napovedi za {selected_date.strftime("%Y-%m-%d")}')
        ax.grid(True)
        ax.legend()
        
        plt.tight_layout()
        
        # Save plot
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {filepath}")
        plt.close()
        
        return fig

    # Create date selection widgets
    num_dates = len(actuals_df.index.unique())
    date_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=num_dates-1,
        step=1,
        description='Date:',
        continuous_update=False,
        layout=widgets.Layout(width='600px')
    )
    
    date_dropdown = widgets.Dropdown(
        options=[(d.strftime('%Y-%m-%d'), i) for i, d in enumerate(actuals_df.index.unique())],
        description='Select date:',
        layout=widgets.Layout(width='300px')
    )
    
    # Create save button and filename input
    filename_input = widgets.Text(
        value='',
        placeholder='Enter filename (optional)',
        description='Filename:',
        layout=widgets.Layout(width='300px')
    )
    
    save_button = widgets.Button(
        description='Save Predictions',
        button_style='success',
        layout=widgets.Layout(width='150px')
    )
    
    save_output = widgets.Output()
    
    def on_save_button_clicked(b):
        save_output.clear_output(wait=True)
        with save_output:
            filename = filename_input.value if filename_input.value else None
            save_predictions_for_date(date_slider.value, arso_pred_df, model_pred_df, actuals_df, filename)
    
    save_button.on_click(on_save_button_clicked)
    
    widgets.jslink((date_slider, 'value'), (date_dropdown, 'index'))
    
    date_output = widgets.Output()
    
    def update_date_plot(change):
        date_output.clear_output(wait=True)
        with date_output:
            plot_predictions_for_date(change['new'], arso_pred_df, model_pred_df, actuals_df, name)
    
    date_slider.observe(update_date_plot, names='value')
    
    with date_output:
        plot_predictions_for_date(date_slider.value, arso_pred_df, model_pred_df, actuals_df, name)
    
    # Display all widgets
    display(widgets.VBox([
        widgets.HBox([date_slider, date_dropdown]),
        widgets.HBox([filename_input, save_button]),
        save_output,
        date_output
    ]))


station_id, name = 4222, "Žiri"
for i, row in runs_df[runs_df['station_id'] == station_id].iterrows():
    model_pred_df = row['model_predictions_df']
    model_pred_df = model_pred_df.set_index('index')
    model_pred_df.index = pd.to_datetime(model_pred_df.index)
    arso_pred_df = row['arso_predictions_df']
    arso_pred_df = arso_pred_df.set_index('index')
    arso_pred_df.index = pd.to_datetime(model_pred_df.index)
    actuals_df = row['actuals_df']
    actuals_df = actuals_df.set_index('index')
    actuals_df.index = pd.to_datetime(model_pred_df.index)

    # Find examples where actual data is >100
    high_flow_examples = actuals_df[actuals_df.max(axis=1) > 100]
    if not high_flow_examples.empty:
        print(f"Found {len(high_flow_examples)} examples where actual flow > 100 m³/s:")
        print(f"Date range: {high_flow_examples.index.min()} to {high_flow_examples.index.max()}")
        print("Sample high flow events:")
        print(high_flow_examples.index)
    else:
        print("No examples found where actual flow > 100 m³/s")

    print(arso_pred_df.shape, model_pred_df.shape, actuals_df.shape)

    create_interactive_plots(arso_pred_df, model_pred_df, actuals_df, name)

    #print(row['mse_scores'])
    #print(row['metrics_table'])
    #print(row['station_id'])
    break

In [ ]:
len(vv_levels['station_id'].unique())

In [ ]:
def analyze_model_improvements(runs_df):
    """
    Analyze model improvements relative to ARSO baseline across all runs.
    
    Args:
        runs_df: DataFrame containing run information with 'metrics_table' column
        
    Returns:
        tuple: (improvement_df, stats_summary, fig) or (None, None, None) if no data
    """
    # Collect all metrics tables and calculate improvements relative to ARSO
    all_metrics = []

    for idx, row in runs_df.iterrows():
        if row['metrics_table'] is not None:
            metrics_df = row['metrics_table'].copy()
            metrics_df['run_name'] = row['name']
            metrics_df['run_id'] = row['id']
            if( metrics_df['ARSO Nash-Sutcliffe Efficiency']<0).any() or (metrics_df['ARSO Nash-Sutcliffe Efficiency'][0] < 0.3):
                print(f"skipping station {row['station_id']} because of negative NSE")
                #print(metrics_df['ARSO Nash-Sutcliffe Efficiency'])
                continue
            all_metrics.append(metrics_df)
    print("Total number of stations taken into account: ", len(all_metrics))
    if all_metrics:
        # Combine all metrics into one DataFrame
        combined_metrics = pd.concat(all_metrics, ignore_index=True)
        
        # Calculate percent improvement/worsening relative to ARSO for each metric
        metric_columns = [col for col in combined_metrics.columns if col not in ['Hour', 'run_name', 'run_id']]
        
        improvement_data = []
        
        # Define metric mappings for comparison with ARSO
        metric_mappings = {
            'Nash-Sutcliffe Efficiency': 'ARSO Nash-Sutcliffe Efficiency',
            'Pearson Correlation Coefficient': 'ARSO Pearson Correlation Coefficient',
            'dQs': 'ARSO dQs',
            'Model RMSE': 'ARSO RMSE'  # Added RMSE mapping
        }
        
        for metric in metric_columns:
            if 'ARSO' in metric:
                continue  # Skip ARSO baseline metrics
                
            # Find corresponding ARSO metric using the mapping
            arso_metric = metric_mappings.get(metric)
            
            if arso_metric is None or arso_metric not in combined_metrics.columns:
                continue
                
            # Calculate percent improvement for each run and hour
            for run_name in combined_metrics['run_name'].unique():
                run_data = combined_metrics[combined_metrics['run_name'] == run_name]
                
                for hour in run_data['Hour'].unique():
                    hour_data = run_data[run_data['Hour'] == hour]
                    
                    if len(hour_data) > 0:
                        model_value = hour_data[metric].iloc[0]
                        arso_value = hour_data[arso_metric].iloc[0]
                        
                        # Calculate percent improvement (positive = better, negative = worse)
                        if arso_value != 0:
                            if 'dQs' in metric:  # For dQs, closer to 1 is better
                                percent_improvement = (abs(arso_value-1)-abs(model_value-1))
                                _test_metric = (abs(arso_value-1)-abs(model_value-1)) / abs(arso_value-1) * 100
                            elif 'RMSE' in metric:  # For RMSE, lower is better
                                percent_improvement = (arso_value - model_value)/arso_value *100  # Positive means improvement
                                _test_metric = (arso_value - model_value) / arso_value * 100
                            else:  # For NSE and correlation, higher is better
                                percent_improvement = (model_value - arso_value)
                                _test_metric = (model_value - arso_value) / arso_value * 100
                            
                            improvement_data.append({
                                'metric': metric,
                                'hour': hour,
                                'run_name': run_name,
                                'percent_improvement': percent_improvement,
                                'model_value': model_value,
                                'arso_value': arso_value,
                                'test_metric': _test_metric
                            })

        # Convert to DataFrame
        improvement_df = pd.DataFrame(improvement_data)

        # Calculate statistics for each metric and hour
        stats_summary = improvement_df.groupby(['metric', 'hour'])['percent_improvement'].agg([
            'mean', 'std', 'min', 'max', 'count'
        ]).reset_index()
        
        # Create visualization
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        # Set up the plotting style
        plt.style.use('default')
        sns.set_palette("husl")
        
        # Get unique metrics
        unique_metrics = improvement_df['metric'].unique()
        
        # Create subplots for each metric
        fig, axes = plt.subplots(len(unique_metrics), 1, figsize=(12, 6*len(unique_metrics)))
        if len(unique_metrics) == 1:
            axes = [axes]
        
        for i, metric in enumerate(unique_metrics):
            metric_data = improvement_df[improvement_df['metric'] == metric]
            metric_stats = stats_summary[stats_summary['metric'] == metric]
            
            ax = axes[i]
            
            # Create smooth bands using percentiles
            hours = metric_stats['hour'].values
            
            # Calculate percentiles for each hour
            percentiles = []
            for hour in hours:
                hour_data = metric_data[metric_data['hour'] == hour]['percent_improvement']
                p10 = hour_data.quantile(0.1)
                p25 = hour_data.quantile(0.25)
                p50 = hour_data.quantile(0.5)  # median
                p75 = hour_data.quantile(0.75)
                p90 = hour_data.quantile(0.9)
                mean_val = hour_data.mean()
                percentiles.append([p10, p25, p50, p75, p90, mean_val])
            
            percentiles = np.array(percentiles)
            
            # Create gradient bands
            # Outermost band (min-max) - lightest
            ax.fill_between(hours, 
                           [metric_data[metric_data['hour'] == h]['percent_improvement'].min() for h in hours],
                           [metric_data[metric_data['hour'] == h]['percent_improvement'].max() for h in hours],
                           alpha=0.2, label='Min-Max range', color='lightblue')
            
            # Middle band (25th-75th percentile) - medium
            ax.fill_between(hours, percentiles[:, 1], percentiles[:, 3], 
                           alpha=0.4, label='25th-75th percentile (IQR)', color='skyblue')
            
            # Plot median line
            ax.plot(hours, percentiles[:, 2], 
                   linewidth=3, color='darkblue', label='Median', marker='o', markersize=4)
            
            # Plot mean line with different style
            ax.plot(hours, percentiles[:, 5], 
                   linewidth=2, color='navy', linestyle='--', label='Mean (dashed)', 
                   marker='s', markersize=3, alpha=0.8)
            
            # Add individual data points with some transparency
            for hour in hours:
                hour_data = metric_data[metric_data['hour'] == hour]['percent_improvement']
                x_jitter = hour + np.random.normal(0, 0.06, len(hour_data))
                ax.scatter(x_jitter, hour_data, alpha=0.3, s=8, color='darkslategray', 
                          label='Individual runs' if hour == hours[0] else "")
            
            # Add horizontal line at 0 (no improvement)
            ax.axhline(y=0, color='red', linestyle='-', alpha=0.7, linewidth=2, 
                      label='No improvement')
            
            # Styling
            ax.set_xlabel('Forecast Hour', fontsize=12, fontweight='bold')
            ax.set_ylabel('Absolute Metric Improvement', fontsize=12, fontweight='bold')
            ax.set_title(f'{metric} - Improvement Relative to ARSO', 
                        fontsize=14, fontweight='bold', pad=20)
            
            # Enhanced legend
            ax.legend(loc='best', frameon=True, fancybox=True, shadow=True, 
                     framealpha=0.9, fontsize=10)
            
            # Enhanced grid
            ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
            ax.set_facecolor('#fafafa')
            
            # Set x-axis to show all hours
            ax.set_xticks(range(1, int(metric_stats['hour'].max()) + 1))
            
            # Add subtle background gradient
            ax.axhspan(ax.get_ylim()[0], 0, alpha=0.05, color='red')  # Below zero
            ax.axhspan(0, ax.get_ylim()[1], alpha=0.05, color='green')  # Above zero
        
        # Overall figure styling
        fig.patch.set_facecolor('white')
        plt.tight_layout(pad=3.0)
        #plt.show()
        
        # Print overall statistics
        print("\nOverall Statistics Across All Metrics and Hours:")
        overall_stats = improvement_df.groupby('metric')['percent_improvement'].agg([
            'mean', 'std', 'min', 'max', 'count'
        ])
        #print(overall_stats)
        
        return improvement_df, stats_summary, fig
        
    else:
        print("No metrics tables found to analyze")
        return None, None, None

# Call the function per station set
station_sets = runs_df['station_id'].apply(lambda x: str(x)[0]).unique()
print(f"Found station sets: {sorted(station_sets)}")

for station_set in sorted(station_sets):
    print(f"\n{'='*60}")
    print(f"ANALYZING STATION SET {station_set}xxx")
    print(f"{'='*60}")
    
    # Filter runs for this station set
    station_set_runs = runs_df[runs_df['station_id'].apply(lambda x: str(x)[0]) == station_set]
    print(f"Number of runs in station set {station_set}xxx: {len(station_set_runs)}")
    print(f"Stations in this set: {sorted(station_set_runs['station_id'].unique())}")
    
    # Analyze improvements for this station set
    print(station_set_runs.shape)
    improvement_df, stats_summary, fig = analyze_model_improvements(station_set_runs)


In [ ]:
feature_selector.n_features_in_, feature_selector.get_feature_names_out()

In [ ]:
########### now lets read all data from LINREG RUNS
import wandb
import numpy as np
import matplotlib.pyplot as plt
# Initialize wandb API
api = wandb.Api()

# Get all runs from the model_experiment project
my_project = "samotorcan-leon-/Final Linear Regression"
runs = api.runs(my_project)  # Updated with correct username format

# Initialize lists to store run information
run_data = []

for run in runs:
    # Extract run information
    run_info = {
        'id': run.id,
        'name': run.name,
        'station_id': run.config.get('station_id', None),
        'state': run.state,
        'config': dict(run.config),
        'summary': dict(run.summary),
        'created_at': run.created_at,
        'duration': run.summary.get('_runtime', 0),
        'tags': run.tags,
    }
    print(run.name)
    run_info['artifacts'] = run.logged_artifacts()

    # Extract model parameters related to weights and biases
    config = run.config
    run_info['hidden_dim'] = config.get('TOK.hidden_dim', config.get('hidden_dim', None))
    run_info['prediction_layer_hidden_dim'] = config.get('TOK.prediction_layer_hidden_dim', config.get('prediction_layer_hidden_dim', None))
    run_info['dropout_rate'] = config.get('TOK.dropout_rate', config.get('dropout_rate', None))
    run_info['learning_rate'] = config.get('lr', config.get('learning_rate', None))
    run_info['weight_decay'] = config.get('weight_decay', None)
    run_info['use_temporal_blocks'] = config.get('TOK.use_temporal_blocks', config.get('use_temporal_blocks', None))
    run_info['block_hours'] = config.get('TOK.block_hours', config.get('block_hours', None))
    
    run_data.append(run_info)

# Convert to DataFrame for easier analysis
runs_df_linreg = pd.DataFrame(run_data)

print(f"Loaded {len(runs_df_linreg)} runs from model_experiment project")
print(f"Run states: {runs_df_linreg['state'].value_counts().to_dict()}")
print(f"Columns available: {list(runs_df_linreg.columns)}")

In [ ]:
# Download metrics tables for all runs
print("Downloading metrics tables for all runs...")
from tqdm import tqdm
tqdm.pandas()

runs_df_linreg['metrics_table'] = runs_df_linreg.progress_apply(lambda row: download_run_metrics(row, 'metrics_table'), axis=1)
runs_df_linreg['mse_scores'] = runs_df_linreg.progress_apply(lambda row: download_run_metrics(row, 'mse_scores'), axis=1)
runs_df_linreg['arso_predictions_df'] = runs_df_linreg.progress_apply(lambda row: download_run_metrics(row, 'arso_predictions_df'), axis=1)
runs_df_linreg['model_predictions_df'] = runs_df_linreg.progress_apply(lambda row: download_run_metrics(row, 'model_predictions_df'), axis=1)
runs_df_linreg['actuals_df'] = runs_df_linreg.progress_apply(lambda row: download_run_metrics(row, 'actuals_df'), axis=1)

# Check results
successful_downloads = runs_df_linreg['metrics_table'].notna().sum()
print(f"Successfully downloaded {successful_downloads} out of {len(runs_df_linreg)} metrics tables")

In [ ]:
# Loop through each row in runs_df_linreg
for i, row in runs_df_linreg.iterrows():
    metrics_table = row['metrics_table']
    model_pred = row['model_predictions_df']
    arso_pred = row['arso_predictions_df'] 
    actuals = row['actuals_df']

    # Calculate RMSE for each hour
    model_rmse = []
    arso_rmse = []

    for hour in range(18):  # Assuming 18 hours of predictions
        hour_str = str(hour)
        if hour_str in model_pred.columns and hour_str in arso_pred.columns and hour_str in actuals.columns:
            model_rmse.append(np.sqrt(np.mean((model_pred[hour_str].to_numpy() - actuals[hour_str].to_numpy())**2)))
            arso_rmse.append(np.sqrt(np.mean((arso_pred[hour_str].to_numpy() - actuals[hour_str].to_numpy())**2)))
    #print(model_rmse)
    # Add RMSE columns to metrics table
    metrics_table['Model RMSE'] = model_rmse
    metrics_table['ARSO RMSE'] = arso_rmse

    # Update the metrics table in runs_df_linreg
    runs_df_linreg.at[i, 'metrics_table'] = metrics_table

# Display first metrics table as example
print(runs_df_linreg.query("station_id==9300").iloc[0]['station_id'])
runs_df_linreg.query("station_id==9300")['metrics_table'].iloc[0]

In [ ]:

def analyze_model_improvements_comparison(runs_df, runs_df_linreg):
    """
    Analyze model improvements relative to ARSO baseline across all runs for both CNN and Linear Regression models.
    Creates side-by-side comparison plots.
    
    Args:
        runs_df: DataFrame containing CNN run information with 'metrics_table' column
        runs_df_linreg: DataFrame containing Linear Regression run information with 'metrics_table' column
        
    Returns:
        tuple: (improvement_df_cnn, improvement_df_linreg, stats_summary_cnn, stats_summary_linreg, fig) or (None, None, None, None, None) if no data
    """
    
    def process_runs_data(runs_df, model_name):
        """Helper function to process runs data for a specific model type"""
        # Collect all metrics tables and calculate improvements relative to ARSO
        all_metrics = []
        upostevane_postaje = []
        for idx, row in runs_df.iterrows():
            if row['metrics_table'] is not None:
                metrics_df = row['metrics_table'].copy()
                metrics_df['run_name'] = row['name']
                metrics_df['run_id'] = row['id']
                if( metrics_df['ARSO Nash-Sutcliffe Efficiency']<0).any() or (metrics_df['ARSO Nash-Sutcliffe Efficiency'][0] < 0.3):
                    print(f"preskačem postajo {row['station_id']} zaradi negativnega NSE ({model_name})")
                    continue
                upostevane_postaje.append(row['station_id'])
                all_metrics.append(metrics_df)
        print(f"Skupno število postaj upoštevanih za {model_name}: {len(all_metrics)}")
        print(f"Postaje upoštevane za {model_name}: {upostevane_postaje}")
        
        if not all_metrics:
            return None, None
            
        # Combine all metrics into one DataFrame
        combined_metrics = pd.concat(all_metrics, ignore_index=True)
        
        # Calculate percent improvement/worsening relative to ARSO for each metric
        metric_columns = [col for col in combined_metrics.columns if col not in ['Hour', 'run_name', 'run_id']]
        
        improvement_data = []
        
        # Define metric mappings for comparison with ARSO
        metric_mappings = {
            'Nash-Sutcliffe Efficiency': 'ARSO Nash-Sutcliffe Efficiency',
            'Pearson Correlation Coefficient': 'ARSO Pearson Correlation Coefficient',
            'dQs': 'ARSO dQs',
            'Model RMSE': 'ARSO RMSE'  # Added RMSE mapping
        }
        
        for metric in metric_columns:
            if 'ARSO' in metric:
                continue  # Skip ARSO baseline metrics
                
            # Find corresponding ARSO metric using the mapping
            arso_metric = metric_mappings.get(metric)
            
            if arso_metric is None or arso_metric not in combined_metrics.columns:
                continue
                
            # Calculate percent improvement for each run and hour
            for run_name in combined_metrics['run_name'].unique():
                run_data = combined_metrics[combined_metrics['run_name'] == run_name]
                
                for hour in run_data['Hour'].unique():
                    hour_data = run_data[run_data['Hour'] == hour]
                    
                    if len(hour_data) > 0:
                        model_value = hour_data[metric].iloc[0]
                        arso_value = hour_data[arso_metric].iloc[0]
                        
                        # Calculate percent improvement (positive = better, negative = worse)
                        if arso_value != 0:
                            if 'dQs' in metric:  # For dQs, closer to 1 is better
                                percent_improvement = (abs(arso_value-1)-abs(model_value-1)) * 100
                                _test_metric = (abs(arso_value-1)-abs(model_value-1)) / abs(arso_value-1) * 100
                            elif 'RMSE' in metric:  # For RMSE, lower is better
                                percent_improvement = (arso_value - model_value)/arso_value *100  # Positive means improvement
                                _test_metric = (arso_value - model_value) / arso_value * 100
                            else:  # For NSE and correlation, higher is better
                                percent_improvement = (model_value - arso_value)
                                _test_metric = (model_value - arso_value) / arso_value * 100
                            
                            improvement_data.append({
                                'metric': metric,
                                'hour': hour,
                                'run_name': run_name,
                                'percent_improvement': percent_improvement,
                                'model_value': model_value,
                                'arso_value': arso_value,
                                'test_metric': _test_metric
                            })

        # Convert to DataFrame
        improvement_df = pd.DataFrame(improvement_data)

        # Calculate statistics for each metric and hour
        stats_summary = improvement_df.groupby(['metric', 'hour'])['percent_improvement'].agg([
            'mean', 'std', 'min', 'max', 'count'
        ]).reset_index()
        
        return improvement_df, stats_summary
    
    # Process both datasets
    improvement_df_cnn, stats_summary_cnn = process_runs_data(runs_df, "CNN")
    improvement_df_linreg, stats_summary_linreg = process_runs_data(runs_df_linreg, "Linearna regresija")
    
    if improvement_df_cnn is None and improvement_df_linreg is None:
        print("Ni najdenih tabel metrik za analizo za noben model")
        return None, None, None, None, None
    
    # Create visualization
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Set up the plotting style
    plt.style.use('default')
    sns.set_palette("husl")
    
    # Increase global font size for all matplotlib elements
    plt.rcParams.update({
        'font.size': 18,           # base font size
        'axes.titlesize': 22,      # title font size
        'axes.labelsize': 20,      # axis label font size
        'xtick.labelsize': 16,     # x tick font size
        'ytick.labelsize': 16,     # y tick font size
        'legend.fontsize': 5,     # legend font size
        'figure.titlesize': 24     # figure title font size
    })
    
    # Get unique metrics from both datasets
    unique_metrics = set()
    if improvement_df_cnn is not None:
        unique_metrics.update(improvement_df_cnn['metric'].unique())
    if improvement_df_linreg is not None:
        unique_metrics.update(improvement_df_linreg['metric'].unique())
    unique_metrics = sorted(list(unique_metrics))
    
    # Create side-by-side subplots for each metric
    fig, axes = plt.subplots(len(unique_metrics), 2, figsize=(22, 7*len(unique_metrics)))
    if len(unique_metrics) == 1:
        axes = axes.reshape(1, -1)
    
    def plot_model_data(ax, improvement_df, stats_summary, metric, model_name, colors):
        """Helper function to plot data for one model"""
        if improvement_df is None or metric not in improvement_df['metric'].unique():
            ax.text(0.5, 0.5, f'Ni podatkov za {model_name}', 
                   horizontalalignment='center', verticalalignment='center', 
                   transform=ax.transAxes, fontsize=22)
            ax.set_title(f'{metric} - {model_name}', fontsize=24, fontweight='bold', pad=24)
            return
            
        metric_data = improvement_df[improvement_df['metric'] == metric]
        metric_stats = stats_summary[stats_summary['metric'] == metric]
        
        # Create smooth bands using percentiles
        hours = metric_stats['hour'].values
        
        # Calculate percentiles for each hour
        percentiles = []
        for hour in hours:
            hour_data = metric_data[metric_data['hour'] == hour]['percent_improvement']
            p10 = hour_data.quantile(0.1)
            p25 = hour_data.quantile(0.25)
            p50 = hour_data.quantile(0.5)  # median
            p75 = hour_data.quantile(0.75)
            p90 = hour_data.quantile(0.9)
            mean_val = hour_data.mean()
            percentiles.append([p10, p25, p50, p75, p90, mean_val])
        
        percentiles = np.array(percentiles)
        
        # Create gradient bands
        # Outermost band (min-max) - lightest
        ax.fill_between(hours, 
                       [metric_data[metric_data['hour'] == h]['percent_improvement'].min() for h in hours],
                       [metric_data[metric_data['hour'] == h]['percent_improvement'].max() for h in hours],
                       alpha=0.2, label='Min-maks razpon', color=colors['light'])
        
        # Middle band (25th-75th percentile) - medium
        ax.fill_between(hours, percentiles[:, 1], percentiles[:, 3], 
                       alpha=0.4, label='25.-75. percentil (IQR)', color=colors['medium'])
        
        # Plot median line
        ax.plot(hours, percentiles[:, 2], 
               linewidth=3, color=colors['dark'], label='Mediana', marker='o', markersize=8)
        
        # Plot mean line with different style
        #ax.plot(hours, percentiles[:, 5], 
        #       linewidth=2, color=colors['darker'], linestyle='--', label='Mean (dashed)', 
        #       marker='s', markersize=3, alpha=0.8)
        
        # Add individual data points with some transparency
        for hour in hours:
            hour_data = metric_data[metric_data['hour'] == hour]['percent_improvement']
            x_jitter = hour + np.random.normal(0, 0.06, len(hour_data))
            ax.scatter(x_jitter, hour_data, alpha=0.3, s=24, color='darkslategray', 
                      label='Posamezne postaje' if hour == hours[0] else "")
        
        # Add horizontal line at 0 (no improvement)
        ax.axhline(y=0, color='red', linestyle='-', alpha=0.7, linewidth=2, 
                  label='Brez izboljšave')
        
        # Styling
        ax.set_xlabel('Napovedna ura', fontsize=20, fontweight='bold')
        if metric == 'Model RMSE':
            ax.set_ylabel('Relativno izboljšanje metrike (%)', fontsize=20, fontweight='bold')
        else:
            ax.set_ylabel('Absolutno izboljšanje metrike', fontsize=20, fontweight='bold')
        
        metric_name_map = {
            'Model RMSE': 'RMSE',
            'Nash-Sutcliffe Efficiency': 'NSE',
            'Pearson Correlation Coefficient': 'PerCorr',
            'dQs': 'dQs'
        }
        ax.set_title(f'{metric_name_map[metric]} - {model_name}', fontsize=24, fontweight='bold', pad=24)
        
        # Enhanced legend with smaller font size and fewer repeated labels
        handles, labels = ax.get_legend_handles_labels()
        # Remove duplicate labels
        from collections import OrderedDict
        by_label = OrderedDict(zip(labels, handles))
        ax.legend(by_label.values(), by_label.keys(), loc='best', frameon=True, fancybox=True, shadow=True, 
                  framealpha=0.9, fontsize=12)  # Reduced fontsize
        
        # Enhanced grid
        ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
        ax.set_facecolor('#fafafa')
        
        # Set x-axis to show all hours
        if len(hours) > 0:
            ax.set_xticks(range(1, int(metric_stats['hour'].max()) + 1))
    
    # Define colors for each model
    cnn_colors = {'light': 'lightblue', 'medium': 'skyblue', 'dark': 'darkblue', 'darker': 'navy'}
    linreg_colors = {'light': 'lightcoral', 'medium': 'salmon', 'dark': 'darkred', 'darker': 'maroon'}
    
    # Plot each metric for both models
    for i, metric in enumerate(unique_metrics):
        # CNN plot (left column)
        plot_model_data(axes[i, 0], improvement_df_cnn, stats_summary_cnn, metric, 'TOK', cnn_colors)
        
        # Linear Regression plot (right column)
        plot_model_data(axes[i, 1], improvement_df_linreg, stats_summary_linreg, metric, 'Linearna regresija', linreg_colors)
        
        # Adjust y-axis to match the larger range between both plots
        y_min_left = axes[i, 0].get_ylim()[0]
        y_max_left = axes[i, 0].get_ylim()[1]
        y_min_right = axes[i, 1].get_ylim()[0]
        y_max_right = axes[i, 1].get_ylim()[1]
        
        # Set both plots to use the larger range
        overall_y_min = min(y_min_left, y_min_right)
        overall_y_max = max(y_max_left, y_max_right)

        metric_ylim_map = {
            'Model RMSE': (-80, 100),
            'Nash-Sutcliffe Efficiency': (-0.6, 0.7),
            'Pearson Correlation Coefficient': (-0.2, 0.4),
            'dQs': (-65, 75)
        }
        overall_y_min, overall_y_max = metric_ylim_map[metric]
        
        axes[i, 0].set_ylim(overall_y_min, overall_y_max)
        axes[i, 1].set_ylim(overall_y_min, overall_y_max)
        
        # Re-add background gradient after y-axis adjustment
        axes[i, 0].axhspan(overall_y_min, 0, alpha=0.05, color='red')  # Below zero
        axes[i, 0].axhspan(0, overall_y_max, alpha=0.05, color='green')  # Above zero
        axes[i, 1].axhspan(overall_y_min, 0, alpha=0.05, color='red')  # Below zero
        axes[i, 1].axhspan(0, overall_y_max, alpha=0.05, color='green')  # Above zero

        
    
    # Overall figure styling
    fig.patch.set_facecolor('white')
    plt.tight_layout(pad=3.0)
    
    # Print overall statistics for both models
    if improvement_df_cnn is not None:
        print("\nSplošne statistike za CNN model:")
        overall_stats_cnn = improvement_df_cnn.groupby('metric')['percent_improvement'].agg([
            'mean', 'std', 'min', 'max', 'count'
        ])
        print(overall_stats_cnn)
    
    if improvement_df_linreg is not None:
        print("\nSplošne statistike za model linearne regresije:")
        overall_stats_linreg = improvement_df_linreg.groupby('metric')['percent_improvement'].agg([
            'mean', 'std', 'min', 'max', 'count'
        ])
        print(overall_stats_linreg)
    
    return improvement_df_cnn, improvement_df_linreg, stats_summary_cnn, stats_summary_linreg, fig

In [ ]:
# Pokliči funkcijo za vsako skupino rek (nabor postaj) - definirano z prvo številko station_id
# Najprej zagotovi, da imata oba dataframe-a obdelane tabele metrik
runs_df = runs_df.copy().dropna(subset=['metrics_table'])
runs_df_linreg = runs_df_linreg.copy().dropna(subset=['metrics_table'])

# Pridobi edinstvene skupine rek iz obeh podatkovnih nizov
station_sets_cnn = runs_df['station_id'].apply(lambda x: str(x)[0]).unique()
station_sets_linreg = runs_df_linreg['station_id'].apply(lambda x: str(x)[0]).unique()
all_station_sets = sorted(set(list(station_sets_cnn) + list(station_sets_linreg)))

print(f"Najdene skupine rek: {all_station_sets}")
print(f"CNN skupine rek: {sorted(station_sets_cnn)}")
print(f"Linearna regresija skupine rek: {sorted(station_sets_linreg)}")

import matplotlib as mpl

# Nastavi večje pisave za vse grafe
mpl.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 5,
    'figure.titlesize': 24
})

for station_set in all_station_sets:
    print(f"\n{'='*80}")
    print(f"ANALIZA SKUPINE REK {station_set}xxx")
    print(f"{'='*80}")
    
    # Filtriraj izvajanja za to skupino rek iz obeh podatkovnih nizov
    station_set_runs_cnn = runs_df[runs_df['station_id'].apply(lambda x: str(x)[0]) == station_set]
    station_set_runs_linreg = runs_df_linreg[runs_df_linreg['station_id'].apply(lambda x: str(x)[0]) == station_set]
    
    print(f"Število CNN izvajanj v skupini rek {station_set}xxx: {len(station_set_runs_cnn)}")
    print(f"Število izvajanj linearne regresije v skupini rek {station_set}xxx: {len(station_set_runs_linreg)}")
    
    if len(station_set_runs_cnn) > 0:
        print(f"CNN postaje v tej skupini: {sorted(station_set_runs_cnn['station_id'].unique())}")
    if len(station_set_runs_linreg) > 0:
        print(f"Postaje linearne regresije v tej skupini: {sorted(station_set_runs_linreg['station_id'].unique())}")
    
    # Preskoči, če ni podatkov za noben model
    if len(station_set_runs_cnn) == 0 and len(station_set_runs_linreg) == 0:
        print(f"Ni podatkov za skupino rek {station_set}xxx, preskačem...")
        continue

    
    # Analiziraj izboljšave za to skupino rek
    improvement_df_cnn, improvement_df_linreg, stats_summary_cnn, stats_summary_linreg, fig = analyze_model_improvements_comparison(
        station_set_runs_cnn, station_set_runs_linreg
    )
    
    set_to_name = {
        '1': 'Hidrološke postaje Pomurja',
        '2': 'Hidrološke postaje Podravja',
        '3': 'Hidrološke postaje glavnega toka reke Save',
        '4': 'Porečje pritokov reke Save',
        '5': 'Porečje Ljubljanice',
        '6': 'Porečje Savinje',
        '7': 'Porečje Krke',
        '8': 'Porečje Soče',
        '9': 'Porečje jadranskih rek',
    }


    # Add title to the figure and save it
    if fig is not None:
        fig.suptitle(f'{set_to_name[station_set]}', fontsize=28, fontweight='bold', y=0.98)
        plt.tight_layout(rect=[0, 0, 1, 0.98])
        
        import os
        # Save the figure to a file
        filename = f'performance_graphs_station_group_{station_set}.png'
        filepath = os.path.join(FIGURES_DIR, filename)
        
        # Save the figure
        fig.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Graf shranjen v: {filepath}")
        
    plt.show()



In [ ]:
len(processed_station_sets)

In [ ]:
def analyze_model_improvements_comparison(runs_df_cnn, runs_df_linreg):
    """
    Analiziraj izboljšave CNN modela v primerjavi z linearno regresijo kot osnovo preko vseh izvajanj.
    
    Args:
        runs_df_cnn: DataFrame z informacijami o CNN izvajanjih z 'metrics_table' stolpcem
        runs_df_linreg: DataFrame z informacijami o izvajanju linearne regresije z 'metrics_table' stolpcem
        
    Returns:
        tuple: (improvement_df_cnn, improvement_df_linreg, stats_summary_cnn, stats_summary_linreg, fig) ali (None, None, None, None, None) če ni podatkov
    """
    # Zberi vse tabele metrik za oba modela
    all_metrics_cnn = []
    all_metrics_linreg = []

    # Obdelaj CNN izvajanja
    for idx, row in runs_df_cnn.iterrows():
        if row['metrics_table'] is not None:
            metrics_df = row['metrics_table'].copy()
            metrics_df['run_name'] = row['name']
            metrics_df['run_id'] = row['id']
            metrics_df['station_id'] = row['station_id']
            if (metrics_df['ARSO Nash-Sutcliffe Efficiency'] < 0).any() or (metrics_df['ARSO Nash-Sutcliffe Efficiency'][0] < 0.3):
                print(f"preskačem CNN postajo {row['station_id']} zaradi negativnega NSE")
                continue
            all_metrics_cnn.append(metrics_df)
    
    # Obdelaj izvajanja linearne regresije
    for idx, row in runs_df_linreg.iterrows():
        if row['metrics_table'] is not None:
            metrics_df = row['metrics_table'].copy()
            metrics_df['run_name'] = row['name']
            metrics_df['run_id'] = row['id']
            metrics_df['station_id'] = row['station_id']
            if (metrics_df['ARSO Nash-Sutcliffe Efficiency'] < 0).any() or (metrics_df['ARSO Nash-Sutcliffe Efficiency'][0] < 0.3):
                print(f"preskačem LinReg postajo {row['station_id']} zaradi negativnega NSE")
                continue
            all_metrics_linreg.append(metrics_df)
    
    print(f"Skupno število CNN postaj, upoštevanih v analizi: {len(all_metrics_cnn)}")
    print(f"Skupno število LinReg postaj, upoštevanih v analizi: {len(all_metrics_linreg)}")
    
    if not all_metrics_cnn and not all_metrics_linreg:
        print("Ni najdenih veljavnih tabel metrik za noben model")
        return None, None, None, None, None
    
    # Združi metrike za vsak model
    combined_metrics_cnn = pd.concat(all_metrics_cnn, ignore_index=True) if all_metrics_cnn else pd.DataFrame()
    combined_metrics_linreg = pd.concat(all_metrics_linreg, ignore_index=True) if all_metrics_linreg else pd.DataFrame()
    
    # Funkcija za izračun izboljšav v primerjavi z osnovo linearne regresije
    def calculate_improvements_vs_linreg(cnn_metrics, linreg_metrics):
        improvement_data = []
        
        # Definiraj metrike za primerjavo v želenem vrstnem redu: RMSE, NSE, Pearson korelacija, dQs
        metric_columns = ['Model RMSE', 'Nash-Sutcliffe Efficiency', 'Pearson Correlation Coefficient', 'dQs']
        
        for metric in metric_columns:
            if metric not in cnn_metrics.columns or metric not in linreg_metrics.columns:
                continue
                
            # Za vsako CNN izvajanje najdi ustrezno LinReg izvajanje po station_id
            for cnn_station in cnn_metrics['station_id'].unique():
                cnn_station_data = cnn_metrics[cnn_metrics['station_id'] == cnn_station]
                linreg_station_data = linreg_metrics[linreg_metrics['station_id'] == cnn_station]
                
                if len(linreg_station_data) == 0:
                    continue  # Ni ustreznih LinReg podatkov za to postajo
                
                for hour in cnn_station_data['Hour'].unique():
                    cnn_hour_data = cnn_station_data[cnn_station_data['Hour'] == hour]
                    linreg_hour_data = linreg_station_data[linreg_station_data['Hour'] == hour]
                    
                    if len(cnn_hour_data) > 0 and len(linreg_hour_data) > 0:
                        cnn_value = cnn_hour_data[metric].iloc[0]
                        linreg_value = linreg_hour_data[metric].iloc[0]
                        
                        # Izračunaj odstotno izboljšavo (pozitivno = CNN boljši, negativno = LinReg boljši)
                        if linreg_value != 0:
                            if 'dQs' in metric:  # Za dQs je bližje 1 bolje
                                percent_improvement = (abs(linreg_value-1) - abs(cnn_value-1)) * 100
                                _test_metric = (abs(linreg_value-1) - abs(cnn_value-1)) / abs(linreg_value-1) * 100
                            elif 'RMSE' in metric:  # Za RMSE je nižje bolje
                                percent_improvement = (linreg_value - cnn_value) / linreg_value * 100  # Pozitivno pomeni CNN izboljšavo
                                _test_metric = (linreg_value - cnn_value) / linreg_value * 100
                            else:  # Za NSE in korelacijo je višje bolje
                                percent_improvement = (cnn_value - linreg_value)
                                _test_metric = (cnn_value - linreg_value) / linreg_value * 100
                            

                        
                            improvement_data.append({
                                'metric': metric if 'RMSE' not in metric else 'RMSE',
                                'hour': hour,
                                'station_id': cnn_station,
                                'run_name': cnn_hour_data['run_name'].iloc[0],
                                'percent_improvement': percent_improvement,
                                'cnn_value': cnn_value,
                                'linreg_value': linreg_value,
                                'test_metric': _test_metric
                            })
        
        return pd.DataFrame(improvement_data)
    
    # Izračunaj izboljšave
    improvement_df_cnn = calculate_improvements_vs_linreg(combined_metrics_cnn, combined_metrics_linreg)
    improvement_df_linreg = pd.DataFrame()  # LinReg je osnova, zato ni potreben izračun izboljšav

    if improvement_df_cnn.empty:
        print("Ni najdenih ujemajočih se parov postaj med CNN in LinReg izvajanji")
        return None, None, None, None, None
    
    # Izračunaj statistike
    stats_summary_cnn = improvement_df_cnn.groupby(['metric', 'hour'])['percent_improvement'].agg([
        'mean', 'std', 'min', 'max', 'count'
    ]).reset_index()
    stats_summary_linreg = pd.DataFrame()  # LinReg je osnova
    
    # Ustvari vizualizacijo
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Nastavi slog risanja
    plt.style.use('default')
    sns.set_palette("husl")
    
    # Nastavi večje globalne font size za grafe
    plt.rcParams.update({
        'font.size': 20,
        'axes.titlesize': 26,
        'axes.labelsize': 22,
        'xtick.labelsize': 18,
        'ytick.labelsize': 18,
        'legend.fontsize': 18,
        'figure.titlesize': 30
    })
    
    # Pridobi edinstvene metrike v želenem vrstnem redu: RMSE, NSE, Pearson korelacija, dQs
    metric_order = ['RMSE', 'Nash-Sutcliffe Efficiency', 'Pearson Correlation Coefficient', 'dQs']
    unique_metrics = [metric for metric in metric_order if metric in improvement_df_cnn['metric'].unique()]
    
    # Ustvari podgrafe za vsako metriko
    fig, axes = plt.subplots(len(unique_metrics), 1, figsize=(22, 7*len(unique_metrics)))
    if len(unique_metrics) == 1:
        axes = [axes]
    
    for i, metric in enumerate(unique_metrics):
        metric_data = improvement_df_cnn[improvement_df_cnn['metric'] == metric]
        metric_stats = stats_summary_cnn[stats_summary_cnn['metric'] == metric]
        
        ax = axes[i]
        
        # Ustvari gladke pasove z uporabo percentilev
        hours = metric_stats['hour'].values
        
        # Izračunaj percentile za vsako uro
        percentiles = []
        for hour in hours:
            hour_data = metric_data[metric_data['hour'] == hour]['percent_improvement']
            p10 = hour_data.quantile(0.1)
            p25 = hour_data.quantile(0.25)
            p50 = hour_data.quantile(0.5)  # mediana
            p75 = hour_data.quantile(0.75)
            p90 = hour_data.quantile(0.9)
            mean_val = hour_data.mean()
            percentiles.append([p10, p25, p50, p75, p90, mean_val])
        
        percentiles = np.array(percentiles)
        
        # Ustvari gradientne pasove
        # Najbolj zunanji pas (min-max) - najsvetlejši
        #ax.fill_between(hours, 
        #               [metric_data[metric_data['hour'] == h]['percent_improvement'].min() for h in hours],
        #               [metric_data[metric_data['hour'] == h]['percent_improvement'].max() for h in hours],
        #               alpha=0.2, label='Min-Max razpon', color='lightblue')
        
        # Srednji pas (25.-75. percentil) - srednji
        # Plot IQR (10th-90th percentile) band with a darker, more visible color
        ax.fill_between(hours, percentiles[:, 0], percentiles[:, 4], 
                       alpha=0.35, label='10.-90. percentil', color='#64b5f6')  # lighter blue

        ax.fill_between(hours, percentiles[:, 1], percentiles[:, 3], 
                       alpha=0.4, label='25.-75. percentil (IQR)', color='#1976d2')  # darker blue
        #ax.plot(hours, percentiles[:, 5], 
        #        linewidth=2, color='orange', linestyle='--', label='Povprečje', 
        #        marker='s', markersize=6, alpha=0.9)
        
        # Plot mean ± std as a band
        #stds = [metric_data[metric_data['hour'] == h]['percent_improvement'].std() for h in hours]
        #means = percentiles[:, 5]
        #ax.fill_between(hours, means - stds, means + stds, 
        #                color='orange', alpha=0.2, label='Povprečje ± std')
        
        # Nariši črto mediane
        ax.plot(hours, percentiles[:, 2], 
               linewidth=3, color='darkblue', label='Mediana', marker='o', markersize=8)
        
        ## Nariši črto povprečja z drugačnim slogom
        #ax.plot(hours, percentiles[:, 5], 
        #       linewidth=2, color='navy', linestyle='--', label='Povprečje (črtkano)', 
        #       marker='s', markersize=3, alpha=0.8)
        
        # Dodaj posamezne podatkovne točke z nekaj prosojnosti
        #for hour in hours:
        #    hour_data = metric_data[metric_data['hour'] == hour]['percent_improvement']
        #    x_jitter = hour + np.random.normal(0, 0.06, len(hour_data))
        #    ax.scatter(x_jitter, hour_data, alpha=0.3, s=20, color='darkslategray', 
        #              label='Posamezne postaje' if hour == hours[0] else "")
        
        # Dodaj vodoravno črto pri 0 (brez izboljšave)
        ax.axhline(y=0, color='red', linestyle='-', alpha=0.7, linewidth=2, 
                  label='Brez izboljšave')
        
        # Oblikovanje
        ax.set_xlabel('Napovedna ura', fontsize=22, fontweight='bold')
        ax.set_xlim(0, int(metric_stats['hour'].max()) + 1)
        if metric == 'RMSE':
            ax.set_ylabel('Relativno izboljšanje metrike (%)', fontsize=22, fontweight='bold')
        elif metric == 'dQs':
            ax.set_ylabel('Absolutno izboljšanje metrike', fontsize=22, fontweight='bold')
        else:
            ax.set_ylabel('Absolutno izboljšanje metrike', fontsize=22, fontweight='bold')

        metric_name_map = {
            'RMSE': 'RMSE',
            'Nash-Sutcliffe Efficiency': 'NSE',
            'Pearson Correlation Coefficient': 'PerCorr',
            'dQs': 'dQs'
        }
        ax.set_title(f'{metric_name_map[metric]} - TOK izboljšava v primerjavi z linearno regresijo', 
                    fontsize=28, fontweight='bold', pad=24)
        
        # Izboljšana legenda
        ax.legend(loc='best', frameon=True, fancybox=True, shadow=True, 
                 framealpha=0.9, fontsize=14)
        
        # Izboljšana mreža
        ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
        ax.set_facecolor('#fafafa')
        
        # Nastavi x-os, da prikaže vse ure
        ax.set_xticks(range(1, int(metric_stats['hour'].max()) + 1))
        
        # Dodaj subtilen gradient ozadja
        #ax.axhspan(ax.get_ylim()[0], 0, alpha=0.05, color='red')  # Pod nič (LinReg boljši)
        #ax.axhspan(0, ax.get_ylim()[1], alpha=0.05, color='green')  # Nad nič (CNN boljši)
    
    # Splošno oblikovanje slike
    fig.patch.set_facecolor('white')
    plt.tight_layout(pad=4.0)

    # Shrani sliko
    import os
    filename = f'TOKvsLinReg.png'
    filepath = os.path.join(FIGURES_DIR, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Graf shranjen v: {filepath}")

    # Izpiši splošne statistike
    print("\nSplošne statistike - CNN v primerjavi z linearno regresijo:")
    overall_stats = improvement_df_cnn.groupby('metric')['percent_improvement'].agg([
        'mean', 'std', 'min', 'max', 'count'
    ])
    print(overall_stats)
    
    return improvement_df_cnn, improvement_df_linreg, stats_summary_cnn, stats_summary_linreg, fig, all_metrics_cnn, all_metrics_linreg

# Pridobi edinstvene skupine rek iz obeh podatkovnih nizov
station_sets_cnn = runs_df['station_id'].apply(lambda x: str(x)[0]).unique()
station_sets_linreg = runs_df_linreg['station_id'].apply(lambda x: str(x)[0]).unique()
all_station_sets = sorted(set(list(station_sets_cnn) + list(station_sets_linreg)))

print(f"Najdene skupine rek: {all_station_sets}")
print(f"CNN skupine rek: {sorted(station_sets_cnn)}")
print(f"Linearna regresija skupine rek: {sorted(station_sets_linreg)}")


    
# Filtriraj izvajanja za to skupino rek iz obeh podatkovnih nizov
station_set_runs_cnn = runs_df
station_set_runs_linreg = runs_df_linreg

print(f"Število CNN izvajanj v skupini rek {station_set}xxx: {len(station_set_runs_cnn)}")
print(f"Število izvajanj linearne regresije v skupini rek {station_set}xxx: {len(station_set_runs_linreg)}")

if len(station_set_runs_cnn) > 0:
    print(f"CNN postaje v tej skupini: {sorted(station_set_runs_cnn['station_id'].unique())}")
if len(station_set_runs_linreg) > 0:
    print(f"Postaje linearne regresije v tej skupini: {sorted(station_set_runs_linreg['station_id'].unique())}")


# Analiziraj izboljšave za to skupino rek
improvement_df_cnn, improvement_df_linreg, stats_summary_cnn, stats_summary_linreg, fig, all_metrics_cnn, all_metrics_linreg = analyze_model_improvements_comparison(
    station_set_runs_cnn, station_set_runs_linreg
)


In [ ]:
improvement_df_cnn.query('percent_improvement < -0.20 and metric == "Nash-Sutcliffe Efficiency" and hour>7')['station_id'].unique()
#improvement_df_cnn.metric.unique()

In [ ]:

# For each id in allids, find the corresponding name and river from the geojson
import json

# Get ids from all_metrics_cnn
allids = list(map(lambda x: int(x['run_name'].iloc[1].split('_')[2]), all_metrics_cnn))
allids = sorted(allids)

with open('hid_arhiv.geojson', 'r', encoding='utf-8') as f:
    gjson = json.load(f)

stationlist = gjson['features']

# Build a mapping from station_id to (name, river)
id_to_name_river = {}
for feature in stationlist:
    props = feature['properties']
    sid = int(props['id']) if 'id' in props else int(props['station_id'])
    id_to_name_river[sid] = (props['name'], props['river'])

# For each id in allids, find and print the name and river
# Output the mapping as a LaTeX table in two columns
num_ids = len(allids)
half = (num_ids + 1) // 2  # split as evenly as possible

left_ids = allids[:half]
right_ids = allids[half:]

# Output the table in multiple smaller tables to fit on one page each
# Let's split into chunks of 20 rows per table (10 per side)
rows_per_table = 10
num_rows = (len(allids) + 2) // 3  # number of rows if filling column-wise

# Pad allids to make it divisible by 3 for column-wise filling
padded_len = num_rows * 3
allids_padded = allids + [None] * (padded_len - len(allids))

# Arrange ids column-wise
columns = [allids_padded[i*num_rows:(i+1)*num_rows] for i in range(3)]

print("\\begin{table}[ht]")
print("\\caption{Seznam hidroloških postaj, uporabljenih v analizi.}")
print("\\label{postaje_tabela}")
print("\\begingroup")
print("\\setlength{\\tabcolsep}{2pt}")
print("\\renewcommand{\\arraystretch}{1.0}")
print("\\tiny")
print("\\begin{tabular}{lll | lll | lll}")
print("\\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} \\\\ \\hline")
counter = 0
river_names_set = set()
for row in range(num_rows):
    row_strs = []
    for col in range(3):
        sid = columns[col][row]
        if sid is not None:
            if sid in id_to_name_river:
                name, river = id_to_name_river[sid]
                river_names_set.add(river)
                name_escaped = name.replace('&', '\\&').replace('_', '\\_')
                river_escaped = river.replace('&', '\\&').replace('_', '\\_')
                row_strs.append(f"{sid} & {name_escaped} & {river_escaped}")
                counter=counter+1
            else:
                row_strs.append(f"{sid} & Not found & Not found")
        else:
            row_strs.append("& & ")
    print(" & ".join(row_strs) + " \\\\")
print("\\end{tabular}")
print("\\endgroup")
print("\\end{table}")
print("\n\\vspace{1em}\n")
#print(counter)
#print(sorted(list(river_names_set)))
#print(len(river_names_set))

In [ ]:
############ Plotting mean value of each metric NSE, PerCorr and dQs for each station in a group of stations
########### y axis is one group, x axis should be two violin plots of CNN and Linear Regression distributions

def plot_metric_distributions_by_river_group(runs_df, runs_df_linreg, hour=18):
    """
    Create box plots comparing CNN and Linear Regression model performance
    for each river group across different metrics, for a specified prediction hour.

    Args:
        runs_df: DataFrame containing CNN run information
        runs_df_linreg: DataFrame containing Linear Regression run information
        hour: int, which prediction hour to use (1-based, e.g. 18 for 18th hour)
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Set global font scale for seaborn and matplotlib
    sns.set_context("talk", font_scale=1.5)
    plt.rcParams.update({
        'font.size': 18,
        'axes.titlesize': 22,
        'axes.labelsize': 20,
        'xtick.labelsize': 16,
        'ytick.labelsize': 16,
        'legend.fontsize': 18,
        'legend.title_fontsize': 20,
        'figure.titlesize': 26
    })

    # Define river group names
    set_to_name = {
        '1': 'Hidrološke postaje Pomurja',
        '2': 'Hidrološke postaje Podravja',
        '3': 'Hidrološke postaje glavnega toka reke Save',
        '4': 'Porečje pritokov reke Save',
        '5': 'Porečje Ljubljanice',
        '6': 'Porečje Savinje',
        '7': 'Porečje Krke',
        '8': 'Porečje Soče',
        '9': 'Porečje jadranskih rek',
    }

    # Get unique river groups from both datasets
    station_sets_cnn = runs_df['station_id'].apply(lambda x: str(x)[0]).unique()
    station_sets_linreg = runs_df_linreg['station_id'].apply(lambda x: str(x)[0]).unique()
    all_station_sets = sorted(set(list(station_sets_cnn) + list(station_sets_linreg)))

    # Define metrics to analyze
    metrics_to_plot = ['Nash-Sutcliffe Efficiency', 'Pearson Correlation Coefficient', 'dQs']
    metric_name_map = {
        'Nash-Sutcliffe Efficiency': 'NSE',
        'Pearson Correlation Coefficient': 'PerCorr',
        'dQs': 'dQs'
    }

    # Define metric mappings for ARSO comparison
    metric_mappings = {
        'Nash-Sutcliffe Efficiency': 'ARSO Nash-Sutcliffe Efficiency',
        'Pearson Correlation Coefficient': 'ARSO Pearson Correlation Coefficient',
        'dQs': 'ARSO dQs',
        'Model RMSE': 'ARSO RMSE'  # Added RMSE mapping
    }

    # Define valid ranges for metrics to clip outliers
    metric_ranges = {
        'NSE': (-1, 1),
        'PerCorr': (-1, 1),
        'dQs': (-100, 900)  # Reasonable range for dQs after transformation
    }

    # Collect data for all river groups and models
    plot_data = []

    for station_set in all_station_sets:
        # Get the proper name for this river group
        river_group_name = set_to_name.get(station_set, f"Skupina {station_set}")

        # Filter runs for this river group
        cnn_runs = runs_df[runs_df['station_id'].apply(lambda x: str(x)[0]) == station_set]
        linreg_runs = runs_df_linreg[runs_df_linreg['station_id'].apply(lambda x: str(x)[0]) == station_set]

        # Process CNN data
        for _, run in cnn_runs.iterrows():
            if run['metrics_table'] is not None:
                metrics_df = run['metrics_table']

                # Filter out stations with poor ARSO performance
                if 'ARSO Nash-Sutcliffe Efficiency' in metrics_df.columns:
                    if (metrics_df['ARSO Nash-Sutcliffe Efficiency'] < 0).any() or (metrics_df['ARSO Nash-Sutcliffe Efficiency'].iloc[0] < 0.3):
                        continue

                # Calculate metrics for the specified hour for each metric
                for metric in metrics_to_plot:
                    if metric in metrics_df.columns:
                        # Take the value for the specified hour (1-based index)
                        if hour <= len(metrics_df):
                            value_idx = hour - 1
                        else:
                            value_idx = -1  # fallback to last hour if out of bounds
                        hour_value = metrics_df[metric].iloc[value_idx]

                        # Transform dQs values: subtract 1 and multiply by 100
                        if metric == 'dQs':
                            hour_value = (hour_value - 1) * 100

                        # Clip values to valid range
                        metric_short = metric_name_map[metric]
                        if metric_short in metric_ranges:
                            min_val, max_val = metric_ranges[metric_short]
                            hour_value = np.clip(hour_value, min_val, max_val)

                        plot_data.append({
                            'river_group': river_group_name,
                            'model': 'TOK',
                            'metric': metric_short,
                            'value': hour_value,
                            'station_id': run['station_id']
                        })

                    # Add ARSO comparison data if available
                    arso_metric = metric_mappings.get(metric)
                    if arso_metric and arso_metric in metrics_df.columns:
                        if hour <= len(metrics_df):
                            value_idx = hour - 1
                        else:
                            value_idx = -1
                        arso_hour_value = metrics_df[arso_metric].iloc[value_idx]

                        # Transform dQs values: subtract 1 and multiply by 100
                        if metric == 'dQs':
                            arso_hour_value = (arso_hour_value - 1) * 100

                        # Clip values to valid range
                        metric_short = metric_name_map[metric]
                        if metric_short in metric_ranges:
                            min_val, max_val = metric_ranges[metric_short]
                            arso_hour_value = np.clip(arso_hour_value, min_val, max_val)

                        plot_data.append({
                            'river_group': river_group_name,
                            'model': 'ARSO',
                            'metric': metric_short,
                            'value': arso_hour_value,
                            'station_id': run['station_id']
                        })

        # Process Linear Regression data
        for _, run in linreg_runs.iterrows():
            if run['metrics_table'] is not None:
                metrics_df = run['metrics_table']

                # Filter out stations with poor ARSO performance
                if 'ARSO Nash-Sutcliffe Efficiency' in metrics_df.columns:
                    if (metrics_df['ARSO Nash-Sutcliffe Efficiency'] < 0).any() or (metrics_df['ARSO Nash-Sutcliffe Efficiency'].iloc[0] < 0.3):
                        continue

                # Calculate metrics for the specified hour for each metric
                for metric in metrics_to_plot:
                    if metric in metrics_df.columns:
                        if hour <= len(metrics_df):
                            value_idx = hour - 1
                        else:
                            value_idx = -1
                        hour_value = metrics_df[metric].iloc[value_idx]

                        # Transform dQs values: subtract 1 and multiply by 100
                        if metric == 'dQs':
                            hour_value = (hour_value - 1) * 100

                        # Clip values to valid range
                        metric_short = metric_name_map[metric]
                        if metric_short in metric_ranges:
                            min_val, max_val = metric_ranges[metric_short]
                            hour_value = np.clip(hour_value, min_val, max_val)

                        plot_data.append({
                            'river_group': river_group_name,
                            'model': 'Linearna regresija',
                            'metric': metric_short,
                            'value': hour_value,
                            'station_id': run['station_id']
                        })

    # Convert to DataFrame
    plot_df = pd.DataFrame(plot_data)

    if plot_df.empty:
        print("Ni podatkov za prikaz")
        return

    # Ensure the model column is categorical and ordered as required: TOK, Linearna regresija, ARSO
    plot_df['model'] = plot_df['model'].replace('ARSO', 'HPS')
    model_order = ['TOK', 'Linearna regresija', 'HPS']
    plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)

    # Create the plot with box plots instead of violin plots in a 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(22, 15))
    axes = axes.flatten()  # Flatten to make indexing easier

    # Add main title to the figure
    fig.suptitle(f"Porazdelitev metrik modelov za {hour}. napovedno uro za posamezne skupine postaj",
                 fontsize=26, fontweight='bold', y=0.95)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    # Store legend info from first plot
    legend_handles = None
    legend_labels = None

    for i, metric in enumerate(['NSE', 'PerCorr', 'dQs']):
        metric_data = plot_df[plot_df['metric'] == metric]

        if not metric_data.empty:
            # Sort model column for plotting order
            metric_data = metric_data.copy()
            metric_data['model'] = pd.Categorical(metric_data['model'], categories=model_order, ordered=True)

            # Create box plot with specified order
            sns.boxplot(
                data=metric_data, x='model', y='value', hue='river_group',
                ax=axes[i], palette='tab10', showfliers=False, order=model_order
            )

            # Set y-axis limits based on metric valid ranges
            if metric in ['NSE']:
                axes[i].set_ylim(0, 1.)
                axes[i].set_yticks(np.linspace(0, 1, 11))
            elif metric in ['PerCorr']:
                axes[i].set_ylim(0.5, 1)
                axes[i].set_yticks(np.linspace(0.5, 1, 11))
            elif metric == 'dQs':
                y_min = max(-100, metric_data['value'].min() * 1.1)//5*5
                y_max = min(900, metric_data['value'].max() * 1.1)//5*5
                #axes[i].set_ylim(y_min, y_max)
                #axes[i].set_yticks(np.linspace(y_min, y_max, 11))

            # Styling
            #if metric =='dQs':
            #    metric = 'dQp' # rename
            axes[i].set_title(f'{metric}', fontsize=22, fontweight='bold')
            axes[i].set_xlabel('Model', fontsize=20, fontweight='bold')
            if metric == 'dQs':
                axes[i].set_ylabel(f'dQs vrednost (%)', fontsize=20, fontweight='bold')
            else:
                axes[i].set_ylabel(f'{metric} vrednost', fontsize=20, fontweight='bold')
            axes[i].grid(True, alpha=0.3)
            axes[i].minorticks_on()
            axes[i].grid(True, which='minor', alpha=0.15)
            axes[i].tick_params(axis='both', which='major', labelsize=16)

            # Store legend info from first plot and remove legend from all plots
            if legend_handles is None:
                handles, labels = axes[i].get_legend_handles_labels()
                n_groups = len(plot_df['river_group'].unique())
                legend_handles = handles[:n_groups]
                legend_labels = labels[:n_groups]

            # Remove legend from individual plots
            axes[i].get_legend().remove()
        else:
            axes[i].text(0.5, 0.5, f'Ni podatkov za {metric}',
                        horizontalalignment='center', verticalalignment='center',
                        transform=axes[i].transAxes, fontsize=20)
            axes[i].set_title(f'{metric}', fontsize=22, fontweight='bold')

    # Hide the fourth subplot (bottom right) and add legend there
    axes[3].set_visible(False)

    # Add legend in the bottom right subplot position
    if legend_handles is not None:
        axes[3].legend(
            legend_handles, legend_labels, title='Skupina postaj',
            loc='center', fontsize=18, title_fontsize=20
        )
        axes[3].set_visible(True)
        axes[3].axis('off')  # Hide axes but keep the subplot visible for legend

    import os
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    filename = f"boxplot_metrike_{hour}.png"
    filepath = os.path.join(FIGURES_DIR, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Graf shranjen v: {filepath}")
    plt.show()

    # Print summary statistics
    summary_stats = plot_df.groupby(['river_group', 'model', 'metric'])['value'].agg([
        'count', "median", 'mean', 'std', 'min', 'max'
    ]).round(4)
    #print(summary_stats)

    return plot_df, summary_stats

# Call the function
plot_data, summary_stats = plot_metric_distributions_by_river_group(runs_df, runs_df_linreg, hour=18)
plot_data, summary_stats = plot_metric_distributions_by_river_group(runs_df, runs_df_linreg, hour=12)
plot_data, summary_stats = plot_metric_distributions_by_river_group(runs_df, runs_df_linreg, hour=6)


In [ ]:

# For each id in allids, find the corresponding name and river from the geojson
import json

# Get ids from all_metrics_cnn
allids = list(map(lambda x: int(x['run_name'].iloc[1].split('_')[2]), all_metrics_cnn))
allids = sorted(allids)

with open('hid_arhiv.geojson', 'r', encoding='utf-8') as f:
    gjson = json.load(f)

stationlist = gjson['features']

# Build a mapping from station_id to (name, river)
id_to_name_river = {}
for feature in stationlist:
    props = feature['properties']
    sid = int(props['id']) if 'id' in props else int(props['station_id'])
    id_to_name_river[sid] = (props['name'], props['river'])

# For each id in allids, find and print the name and river
# Output the mapping as a LaTeX table in two columns
num_ids = len(allids)
half = (num_ids + 1) // 2  # split as evenly as possible

left_ids = allids[:half]
right_ids = allids[half:]

# Output the table in multiple smaller tables to fit on one page each
# Let's split into chunks of 20 rows per table (10 per side)
rows_per_table = 10
num_rows = (len(allids) + 2) // 3  # number of rows if filling column-wise

# Pad allids to make it divisible by 3 for column-wise filling
padded_len = num_rows * 3
allids_padded = allids + [None] * (padded_len - len(allids))

# Arrange ids column-wise
columns = [allids_padded[i*num_rows:(i+1)*num_rows] for i in range(3)]

print("\\begin{table}[ht]")
print("\\caption{Seznam hidroloških postaj, uporabljenih v analizi.}")
print("\\label{postaje_tabela}")
print("\\begingroup")
print("\\setlength{\\tabcolsep}{2pt}")
print("\\renewcommand{\\arraystretch}{1.0}")
print("\\tiny")
print("\\begin{tabular}{lll | lll | lll}")
print("\\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} \\\\ \\hline")
counter = 0
river_names_set = set()
for row in range(num_rows):
    row_strs = []
    for col in range(3):
        sid = columns[col][row]
        if sid is not None:
            if sid in id_to_name_river:
                name, river = id_to_name_river[sid]
                river_names_set.add(river)
                name_escaped = name.replace('&', '\\&').replace('_', '\\_')
                river_escaped = river.replace('&', '\\&').replace('_', '\\_')
                row_strs.append(f"{sid} & {name_escaped} & {river_escaped}")
                counter=counter+1
            else:
                row_strs.append(f"{sid} & Not found & Not found")
        else:
            row_strs.append("& & ")
    print(" & ".join(row_strs) + " \\\\")
print("\\end{tabular}")
print("\\endgroup")
print("\\end{table}")
print("\n\\vspace{1em}\n")
#print(counter)
#print(sorted(list(river_names_set)))
#print(len(river_names_set))

In [ ]:

# For each id in allids, find the corresponding name and river from the geojson
import json

# Get ids from all_metrics_cnn
allids = list(map(lambda x: int(x['run_name'].iloc[1].split('_')[2]), all_metrics_cnn))
allids = sorted(allids)

with open('hid_arhiv.geojson', 'r', encoding='utf-8') as f:
    gjson = json.load(f)

stationlist = gjson['features']

# Build a mapping from station_id to (name, river)
id_to_name_river = {}
for feature in stationlist:
    props = feature['properties']
    sid = int(props['id']) if 'id' in props else int(props['station_id'])
    id_to_name_river[sid] = (props['name'], props['river'])

# For each id in allids, find and print the name and river
# Output the mapping as a LaTeX table in two columns
num_ids = len(allids)
half = (num_ids + 1) // 2  # split as evenly as possible

left_ids = allids[:half]
right_ids = allids[half:]

# Output the table in multiple smaller tables to fit on one page each
# Let's split into chunks of 20 rows per table (10 per side)
rows_per_table = 10
num_rows = (len(allids) + 2) // 3  # number of rows if filling column-wise

# Pad allids to make it divisible by 3 for column-wise filling
padded_len = num_rows * 3
allids_padded = allids + [None] * (padded_len - len(allids))

# Arrange ids column-wise
columns = [allids_padded[i*num_rows:(i+1)*num_rows] for i in range(3)]

print("\\begin{table}[ht]")
print("\\caption{Seznam hidroloških postaj, uporabljenih v analizi.}")
print("\\label{postaje_tabela}")
print("\\begingroup")
print("\\setlength{\\tabcolsep}{2pt}")
print("\\renewcommand{\\arraystretch}{1.0}")
print("\\tiny")
print("\\begin{tabular}{lll | lll | lll}")
print("\\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} \\\\ \\hline")
counter = 0
river_names_set = set()
for row in range(num_rows):
    row_strs = []
    for col in range(3):
        sid = columns[col][row]
        if sid is not None:
            if sid in id_to_name_river:
                name, river = id_to_name_river[sid]
                river_names_set.add(river)
                name_escaped = name.replace('&', '\\&').replace('_', '\\_')
                river_escaped = river.replace('&', '\\&').replace('_', '\\_')
                row_strs.append(f"{sid} & {name_escaped} & {river_escaped}")
                counter=counter+1
            else:
                row_strs.append(f"{sid} & Not found & Not found")
        else:
            row_strs.append("& & ")
    print(" & ".join(row_strs) + " \\\\")
print("\\end{tabular}")
print("\\endgroup")
print("\\end{table}")
print("\n\\vspace{1em}\n")
#print(counter)
#print(sorted(list(river_names_set)))
#print(len(river_names_set))

In [ ]:

# For each id in allids, find the corresponding name and river from the geojson
import json

# Get ids from all_metrics_cnn
allids = list(map(lambda x: int(x['run_name'].iloc[1].split('_')[2]), all_metrics_cnn))
allids = sorted(allids)

with open('hid_arhiv.geojson', 'r', encoding='utf-8') as f:
    gjson = json.load(f)

stationlist = gjson['features']

# Build a mapping from station_id to (name, river)
id_to_name_river = {}
for feature in stationlist:
    props = feature['properties']
    sid = int(props['id']) if 'id' in props else int(props['station_id'])
    id_to_name_river[sid] = (props['name'], props['river'])

# For each id in allids, find and print the name and river
# Output the mapping as a LaTeX table in two columns
num_ids = len(allids)
half = (num_ids + 1) // 2  # split as evenly as possible

left_ids = allids[:half]
right_ids = allids[half:]

# Output the table in multiple smaller tables to fit on one page each
# Let's split into chunks of 20 rows per table (10 per side)
rows_per_table = 10
num_rows = (len(allids) + 2) // 3  # number of rows if filling column-wise

# Pad allids to make it divisible by 3 for column-wise filling
padded_len = num_rows * 3
allids_padded = allids + [None] * (padded_len - len(allids))

# Arrange ids column-wise
columns = [allids_padded[i*num_rows:(i+1)*num_rows] for i in range(3)]

print("\\begin{table}[ht]")
print("\\caption{Seznam hidroloških postaj, uporabljenih v analizi.}")
print("\\label{postaje_tabela}")
print("\\begingroup")
print("\\setlength{\\tabcolsep}{2pt}")
print("\\renewcommand{\\arraystretch}{1.0}")
print("\\tiny")
print("\\begin{tabular}{lll | lll | lll}")
print("\\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} & \\textbf{ID} & \\textbf{Ime} & \\textbf{Reka} \\\\ \\hline")
counter = 0
river_names_set = set()
for row in range(num_rows):
    row_strs = []
    for col in range(3):
        sid = columns[col][row]
        if sid is not None:
            if sid in id_to_name_river:
                name, river = id_to_name_river[sid]
                river_names_set.add(river)
                name_escaped = name.replace('&', '\\&').replace('_', '\\_')
                river_escaped = river.replace('&', '\\&').replace('_', '\\_')
                row_strs.append(f"{sid} & {name_escaped} & {river_escaped}")
                counter=counter+1
            else:
                row_strs.append(f"{sid} & Not found & Not found")
        else:
            row_strs.append("& & ")
    print(" & ".join(row_strs) + " \\\\")
print("\\end{tabular}")
print("\\endgroup")
print("\\end{table}")
print("\n\\vspace{1em}\n")
#print(counter)
#print(sorted(list(river_names_set)))
#print(len(river_names_set))

In [ ]:
summary_stats.query('river_group == "Porečje jadranskih rek"')

In [ ]:
# Plot metric distributions for all stations at once, for every hour

def plot_metric_distributions_all_stations_by_hour(runs_df, runs_df_linreg):
    """
    Create box plots comparing CNN and Linear Regression model performance
    for all stations (not split by river group), for every prediction hour.
    Each metric is plotted in a separate row (vertical layout).
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd

    metrics_to_plot = ['Nash-Sutcliffe Efficiency', 'Pearson Correlation Coefficient', 'dQs']
    metric_name_map = {
        'Nash-Sutcliffe Efficiency': 'NSE',
        'Pearson Correlation Coefficient': 'PerCorr',
        'dQs': 'dQs'
    }
    metric_ranges = {
        'NSE': (-1, 1),
        'PerCorr': (-1, 1),
        'dQs': (-100, 900)
    }
    metric_mappings = {
        'Nash-Sutcliffe Efficiency': 'ARSO Nash-Sutcliffe Efficiency',
        'Pearson Correlation Coefficient': 'ARSO Pearson Correlation Coefficient',
        'dQs': 'ARSO dQs',
        'Model RMSE': 'ARSO RMSE'
    }

    plot_data = []

    # Helper to process a run (CNN or LinReg)
    def process_run(run, model_name):
        metrics_df = run['metrics_table']
        if metrics_df is None:
            return
        # Filter out stations with poor ARSO performance
        if 'ARSO Nash-Sutcliffe Efficiency' in metrics_df.columns:
            if (metrics_df['ARSO Nash-Sutcliffe Efficiency'] < 0).any() or (metrics_df['ARSO Nash-Sutcliffe Efficiency'].iloc[0] < 0.3):
                return
        for metric in metrics_to_plot:
            if metric in metrics_df.columns:
                metric_short = metric_name_map[metric]
                for hour_idx in range(len(metrics_df)):
                    value = metrics_df[metric].iloc[hour_idx]
                    if metric == 'dQs':
                        value = (value - 1) * 100
                    if metric_short in metric_ranges:
                        min_val, max_val = metric_ranges[metric_short]
                        value = np.clip(value, min_val, max_val)
                    plot_data.append({
                        'model': model_name,
                        'metric': metric_short,
                        'value': value,
                        'station_id': run['station_id'],
                        'hour': hour_idx + 1  # assuming 1-based hour
                    })
            # Add ARSO comparison if available
            arso_metric = metric_mappings.get(metric)
            if arso_metric and arso_metric in metrics_df.columns:
                metric_short = metric_name_map[metric]
                for hour_idx in range(len(metrics_df)):
                    value = metrics_df[arso_metric].iloc[hour_idx]
                    if metric == 'dQs':
                        value = (value - 1) * 100
                    if metric_short in metric_ranges:
                        min_val, max_val = metric_ranges[metric_short]
                        value = np.clip(value, min_val, max_val)
                    plot_data.append({
                        'model': 'HPS',
                        'metric': metric_short,
                        'value': value,
                        'station_id': run['station_id'],
                        'hour': hour_idx + 1
                    })

    for _, run in runs_df.iterrows():
        process_run(run, 'TOK')
    for _, run in runs_df_linreg.iterrows():
        process_run(run, 'Linearna regresija')

    plot_df = pd.DataFrame(plot_data)
    print(plot_df.groupby('model').count())
    if plot_df.empty:
        print("Ni podatkov za prikaz")
        return

    # Set the order of models for plotting: TOK, Linearna regresija, HPS
    model_order = ['TOK', 'Linearna regresija', 'HPS']
    plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)

    # Plot for each metric: x=hour, y=value, hue=model (all stations together)
    sns.set_context("talk", font_scale=1.5)
    sns.set_context("talk", font_scale=1.2)
    plt.rcParams.update({
        'font.size': 14,
        'axes.titlesize': 16,
        'axes.labelsize': 15,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 13,
        'legend.title_fontsize': 15,
        'figure.titlesize': 18
    })
    # Change: 3 rows, 1 column
    fig, axes = plt.subplots(3, 1, figsize=(14, 20))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for i, metric in enumerate(['NSE', 'PerCorr', 'dQs']):
        metric_data = plot_df[plot_df['metric'] == metric]
        ax = axes[i]
        if not metric_data.empty:
            sns.boxplot(
                data=metric_data,
                x='hour',
                y='value',
                hue='model',
                ax=ax,
                palette='tab10',
                showfliers=False,
                hue_order=model_order
            )
            #if metric == 'dQs':
            #    metric = 'dQp'
            ax.set_title(f'{metric} za vse postaje', fontsize=18, fontweight='bold')
            ax.set_xlabel('Napovedna ura', fontsize=16)
            if metric == 'dQs':
                ax.set_ylabel(f'{metric} vrednost (%)', fontsize=16)
            else:
                ax.set_ylabel(f'{metric} vrednost', fontsize=16)
            if metric == 'NSE':
                ax.set_ylim(0, 1.)
            elif metric == 'PerCorr':
                ax.set_ylim(0.5, 1)
            elif metric == 'dQs':
                y_min = max(-100, metric_data['value'].min() * 1.1)
                #y_max = min(900, metric_data['value'].max() * 1.1)
                y_max = 50
                ax.set_ylim(y_min, y_max)
            ax.grid(True, alpha=0.3)
            # Only show main ticks on x and y axes (no minor ticks)
            #ax.minorticks_off()
            # Set x-ticks to only the available hours (main ticks)
            hours_sorted = sorted(metric_data['hour'].unique())
            print(hours_sorted)
            ax.set_xticks([0]+hours_sorted)
            # Optionally, set y-ticks to main ticks only (let matplotlib decide, or set manually if needed)
            # ax.set_yticks(ax.get_yticks())  # This line is not strictly needed, as main ticks are default
            #ax.tick_params(axis='both', which='major', labelsize=16)
            if i == 0:
                ax.legend(title='Model', fontsize=16, title_fontsize=18)
            else:
                ax.get_legend().remove()
        else:
            ax.text(0.5, 0.5, f'Ni podatkov za {metric}',
                         horizontalalignment='center', verticalalignment='center',
                         transform=ax.transAxes, fontsize=20)
            ax.set_title(f'{metric}', fontsize=22, fontweight='bold')

    # Save the figure to a file
    import os
    filename = f'primerjava_modelov_po_urah.png'
    filepath = os.path.join(FIGURES_DIR, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Graf shranjen v: {filepath}")

    plt.show()

    # Print summary statistics
    summary_stats = plot_df.groupby(['hour', 'model', 'metric'])['value'].agg([
        'count', "median",'mean', 'std', 'min', 'max'
    ]).round(4)
    print(summary_stats)
    return plot_df, summary_stats

# Call the function
plot_data_all_hours, summary_stats_all_hours = plot_metric_distributions_all_stations_by_hour(runs_df, runs_df_linreg)